In [ ]:
# Install yfinance
!pip install yfinance --quiet

import yfinance as yf
import pandas as pd

# ✅ NIFTY50 tickers list (latest, NSE suffix needed)
nifty50_tickers = [
    "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS", "ASIANPAINT.NS", "AXISBANK.NS",
    "BAJAJ-AUTO.NS", "BAJFINANCE.NS", "BAJAJFINSV.NS", "BHARTIARTL.NS", "BPCL.NS",
    "BRITANNIA.NS", "CIPLA.NS", "COALINDIA.NS", "DIVISLAB.NS", "DRREDDY.NS",
    "EICHERMOT.NS", "GRASIM.NS", "HCLTECH.NS", "HDFCBANK.NS", "HDFCLIFE.NS",
    "HEROMOTOCO.NS", "HINDALCO.NS", "HINDUNILVR.NS", "ICICIBANK.NS", "INDUSINDBK.NS",
    "INFY.NS", "ITC.NS", "JSWSTEEL.NS", "KOTAKBANK.NS", "LT.NS",
    "M&M.NS", "MARUTI.NS", "NESTLEIND.NS", "NTPC.NS", "ONGC.NS",
    "POWERGRID.NS", "RELIANCE.NS", "SBILIFE.NS", "SBIN.NS", "SUNPHARMA.NS",
    "TATACONSUM.NS", "TATAMOTORS.NS", "TATASTEEL.NS", "TCS.NS", "TECHM.NS",
    "TITAN.NS", "ULTRACEMCO.NS", "UPL.NS", "WIPRO.NS"
]

print(f"Total tickers: {len(nifty50_tickers)}")

# Download all NIFTY50 stock data
data = yf.download(nifty50_tickers, start="2010-01-01", end="2025-01-01", group_by="ticker")

# Reshape into clean ML-friendly format
all_data = []

for ticker in nifty50_tickers:
    if ticker not in data.columns.levels[0]:
        print(f"⚠️ Skipping {ticker} (not available)")
        continue
    df = data[ticker].copy()
    df["Ticker"] = ticker.replace(".NS", "")
    df.reset_index(inplace=True)
    all_data.append(df)

# Merge all stocks into one DataFrame
final_df = pd.concat(all_data, axis=0)
final_df.rename(columns={
    "Adj Close": "Adj_Close"
}, inplace=True)

# Save to CSV
final_df.to_csv("nifty50_all_stocks.csv", index=False)

print("✅ Clean NIFTY50 stock data saved as nifty50_all_stocks.csv")
print(final_df.head(10))


In [ ]:
import pandas as pd
import re

# =========================
# CONFIG
# =========================
STOCK_CSV = "nifty50_all_stocks.csv"        # your NIFTY50 file
HEADLINES_CSV = "india-news-headlines.csv"  # Kaggle Times of India file
OUTPUT_CSV = "merged_nifty50_company_headlines.csv"
CHUNKSIZE = 300_000
KEEP_ONLY_MATCHED = False
DEBUG_MODE = True  # Set to True to see what's happening

# =========================
# 1) LOAD & CLEAN STOCKS
# =========================
stocks = pd.read_csv(STOCK_CSV, dtype={"Date": "string", "Ticker": "string"}, low_memory=False)

# Parse dd-mm-yyyy
stocks["Date"] = pd.to_datetime(stocks["Date"].str.strip(), format="%d-%m-%Y", errors="coerce")
stocks = stocks.dropna(subset=["Date"]).copy()
stocks["Date"] = stocks["Date"].dt.normalize()

# Clean tickers
stocks["Ticker"] = stocks["Ticker"].str.upper().str.replace(".NS", "", regex=False).str.strip()

# Keep only useful columns
keep_cols = [c for c in ["Date", "Ticker", "Open", "High", "Low", "Close", "Adj_Close", "Volume"] if c in stocks.columns]
stocks = stocks[keep_cols]

print("✅ Stocks:", stocks.shape, "| date range:", stocks["Date"].min().date(), "→", stocks["Date"].max().date())
unique_tickers = sorted(stocks["Ticker"].unique())
stock_dates = set(stocks["Date"].dt.date.unique())

# =========================
# 2) EXPANDED TICKER → KEYWORD MAP
# =========================
TICKER_KEYWORDS = {
    "ADANIENT": ["ADANI ENTERPRISES", "ADANIENT", "ADANI ENT", "ADANI GROUP", "ADANI"],
    "ADANIPORTS": ["ADANI PORTS", "APSEZ", "ADANI PORT", "ADANI GROUP", "ADANI"],
    "APOLLOHOSP": ["APOLLO HOSPITALS", "APOLLOHOSP", "APOLLO HOSPITAL", "APOLLO"],
    "ASIANPAINT": ["ASIAN PAINTS", "ASIANPAINT", "ASIAN PAINT"],
    "AXISBANK": ["AXIS BANK", "AXISBANK", "AXIS"],
    "BAJAJ-AUTO": ["BAJAJ AUTO", "BAJAJ-AUTO", "BAJAJ"],
    "BAJFINANCE": ["BAJAJ FINANCE", "BAJFINANCE", "BAJAJ FIN", "BAJAJ"],
    "BAJAJFINSV": ["BAJAJ FINSERV", "BAJAJFINSV", "BAJAJ"],
    "BHARTIARTL": ["BHARTI AIRTEL", "AIRTEL", "BHARTIARTL", "BHARTI"],
    "BPCL": ["BPCL", "BHARAT PETROLEUM", "BHARAT PETROL"],
    "BRITANNIA": ["BRITANNIA"],
    "CIPLA": ["CIPLA"],
    "COALINDIA": ["COAL INDIA", "COALINDIA", "CIL"],
    "DIVISLAB": ["DIVIS LABS", "DIVI'S LABS", "DIVI'S", "DIVIS", "DIVI"],
    "DRREDDY": ["DR REDDY", "DR. REDDY", "DR REDDY'S", "DR REDDY'S", "DRREDDY", "REDDY"],
    "EICHERMOT": ["EICHER MOTORS", "EICHERMOT", "EICHER"],
    "GRASIM": ["GRASIM"],
    "HCLTECH": ["HCL TECH", "HCL TECHNOLOGIES", "HCLTECH", "HCL"],
    "HDFCBANK": ["HDFC BANK", "HDFCBANK", "HDFC"],
    "HDFCLIFE": ["HDFC LIFE", "HDFCLIFE", "HDFC"],
    "HEROMOTOCO": ["HERO MOTOCORP", "HEROMOTOCO", "HERO MOTO", "HERO"],
    "HINDALCO": ["HINDALCO"],
    "HINDUNILVR": ["HINDUSTAN UNILEVER", "HINDUNILVR", "HUL", "UNILEVER"],
    "ICICIBANK": ["ICICI BANK", "ICICIBANK", "ICICI"],
    "INDUSINDBK": ["INDUSIND BANK", "INDUSINDBK", "INDUSIND"],
    "INFY": ["INFOSYS", "INFY"],
    "ITC": ["ITC"],
    "JSWSTEEL": ["JSW STEEL", "JSWSTEEL", "JSW"],
    "KOTAKBANK": ["KOTAK MAHINDRA BANK", "KOTAK BANK", "KOTAKBANK", "KOTAK MAHINDRA", "KOTAK"],
    "LT": ["LARSEN & TOUBRO", "LARSEN AND TOUBRO", "L&T", "LT", "LARSEN TOUBRO"],
    "M&M": ["MAHINDRA & MAHINDRA", "MAHINDRA AND MAHINDRA", "M&M", "MAHINDRA"],
    "MARUTI": ["MARUTI", "MARUTI SUZUKI"],
    "NESTLEIND": ["NESTLE INDIA", "NESTLEIND", "NESTLÉ INDIA", "NESTLE"],
    "NTPC": ["NTPC"],
    "ONGC": ["ONGC", "OIL AND NATURAL GAS", "OIL NATURAL GAS"],
    "POWERGRID": ["POWER GRID", "POWERGRID", "PGCIL"],
    "RELIANCE": ["RELIANCE INDUSTRIES", "RELIANCE", "RIL", "RELIANCE IND"],
    "SBILIFE": ["SBI LIFE", "SBILIFE", "STATE BANK LIFE"],
    "SBIN": ["STATE BANK OF INDIA", "SBI", "SBIN", "STATE BANK"],
    "SUNPHARMA": ["SUN PHARMA", "SUN PHARMACEUTICAL", "SUNPHARMA", "SUN PHARMA INDUSTRIES"],
    "TCS": ["TATA CONSULTANCY SERVICES", "TCS", "TATA CONSULTANCY"],
    "TATACONSUM": ["TATA CONSUMER", "TATA GLOBAL BEVERAGES", "TATACONSUM", "TATA CONSUMER PRODUCTS"],
    "TATAMOTORS": ["TATA MOTORS", "TATAMOTORS", "TATA MOTOR"],
    "TATASTEEL": ["TATA STEEL", "TATASTEEL"],
    "TECHM": ["TECH MAHINDRA", "TECHM"],
    "TITAN": ["TITAN COMPANY", "TITAN"],
    "ULTRACEMCO": ["ULTRATECH CEMENT", "ULTRACEMCO", "ULTRATECH"],
    "UPL": ["UPL", "UNITED PHOSPHORUS"],
    "WIPRO": ["WIPRO"],
}

# Build more flexible regex patterns
PATTERNS = {}
for t in unique_tickers:
    base = t.upper().strip()
    terms = TICKER_KEYWORDS.get(base, [base])
    esc_terms = []
    for term in terms:
        term = term.upper().replace("'", "'")
        # Make regex more flexible - allow for partial matches and common variations
        esc = re.escape(term)
        esc = esc.replace("\\'", "['']")  # allow both ' and '
        esc = esc.replace("\\ ", "\\s+")  # allow multiple spaces
        esc_terms.append(esc)
    # Use word boundaries but make them more flexible
    pattern_str = r"(?:\b(" + "|".join(esc_terms) + r")\b)"
    PATTERNS[base] = re.compile(pattern_str, flags=re.IGNORECASE)

# =========================
# 3) STREAM HEADLINES, FILTER BY DATE & COMPANY
# =========================
detected_text_col = None
agg_parts = []
total_headlines_processed = 0
total_matches_found = 0
sample_headlines = []

reader = pd.read_csv(
    HEADLINES_CSV,
    engine="python",
    on_bad_lines="skip",
    dtype={"publish_date": "string"},
    chunksize=CHUNKSIZE,
)

for i, chunk in enumerate(reader, start=1):
    # detect text column - prioritize actual text columns over categories
    if detected_text_col is None:
        # First, try to find columns that likely contain actual headline text
        priority_columns = []
        for c in chunk.columns:
            cl = c.lower()
            if "headline_text" in cl or "title_text" in cl:
                priority_columns.append((c, 1))  # highest priority
            elif "headline" in cl and "text" in cl:
                priority_columns.append((c, 2))
            elif "title" in cl and not "category" in cl:
                priority_columns.append((c, 3))
            elif cl.endswith("text") and cl != "publish_date" and "category" not in cl:
                priority_columns.append((c, 4))
            elif "headline" in cl and "category" not in cl:
                priority_columns.append((c, 5))  # lower priority for generic headline columns

        if priority_columns:
            # Sort by priority and pick the best one
            priority_columns.sort(key=lambda x: x[1])
            detected_text_col = priority_columns[0][0]
        else:
            raise ValueError(f"Couldn't find a headline text column in: {chunk.columns.tolist()}")
        print(f"📰 Using headline text column: {detected_text_col}")
        print(f"📰 Available columns: {chunk.columns.tolist()}")

    # parse yyyymmdd
    d = pd.to_datetime(chunk["publish_date"].astype(str), format="%Y%m%d", errors="coerce")
    chunk = chunk.loc[d.notna()].copy()
    chunk["Date"] = d.dt.normalize()

    # filter by stock dates
    chunk = chunk[chunk["Date"].dt.date.isin(stock_dates)]
    if chunk.empty:
        continue

    # keep minimal cols
    chunk = chunk[["Date", detected_text_col]].copy()
    chunk[detected_text_col] = chunk[detected_text_col].astype(str)

    total_headlines_processed += len(chunk)

    # Collect sample headlines for debugging
    if DEBUG_MODE and len(sample_headlines) < 10:
        sample_headlines.extend(chunk[detected_text_col].head(5).tolist())

    # company matching
    chunk_matches = 0
    for t in unique_tickers:
        pat = PATTERNS[t]
        mask = chunk[detected_text_col].str.contains(pat, na=False)

        if mask.any():
            chunk_matches += mask.sum()

            if DEBUG_MODE and chunk_matches <= 5:  # Show first few matches
                matches = chunk.loc[mask, detected_text_col].head(2).tolist()
                print(f"🔍 Found {mask.sum()} matches for {t}: {matches}")

        if not mask.any():
            continue

        tmp = chunk.loc[mask, ["Date", detected_text_col]].copy()
        tmp["Ticker"] = t

        # aggregate within chunk
        tmp = (
            tmp.groupby(["Date", "Ticker"], as_index=False)
            .agg(Headlines=(detected_text_col, lambda s: " || ".join(s)),
                 HeadlinesCount=(detected_text_col, "size"))
        )
        agg_parts.append(tmp)

    total_matches_found += chunk_matches

    if i % 5 == 0:
        print(f"…processed {i} chunks; headlines: {total_headlines_processed}, matches: {total_matches_found}")

# Debug information
if DEBUG_MODE:
    print(f"\n📊 DEBUGGING INFO:")
    print(f"Total headlines processed: {total_headlines_processed}")
    print(f"Total matches found: {total_matches_found}")
    print(f"Sample headlines:")
    for j, headline in enumerate(sample_headlines[:5], 1):
        print(f"  {j}. {headline}")

    # Test a few patterns manually
    print(f"\n🧪 TESTING PATTERNS:")
    test_headlines = sample_headlines[:3] if sample_headlines else ["Reliance Industries reports strong quarterly results", "HDFC Bank announces new branch", "Infosys wins major contract"]
    for headline in test_headlines:
        print(f"\nHeadline: {headline}")
        for ticker in ["RELIANCE", "HDFCBANK", "INFY"][:3]:  # Test a few
            if ticker in PATTERNS:
                match = PATTERNS[ticker].search(headline)
                print(f"  {ticker}: {'✓ MATCH' if match else '✗ no match'}")

# combine aggregates
if agg_parts:
    news_by_day_ticker = pd.concat(agg_parts, ignore_index=True)
    news_by_day_ticker = (
        news_by_day_ticker
        .groupby(["Date", "Ticker"], as_index=False)
        .agg(Headlines=("Headlines", lambda s: " || ".join(s)),
             HeadlinesCount=("HeadlinesCount", "sum"))
    )

    # CRITICAL FIX: Ensure Date column is normalized datetime
    news_by_day_ticker["Date"] = pd.to_datetime(news_by_day_ticker["Date"]).dt.normalize()

else:
    news_by_day_ticker = pd.DataFrame(columns=["Date", "Ticker", "Headlines", "HeadlinesCount"])

print("🧮 Company-specific news rows:", news_by_day_ticker.shape)

# =========================
# 4) MERGE WITH STOCKS + SAVE
# =========================
# Ensure both DataFrames have consistent Date types
stocks["Date"] = pd.to_datetime(stocks["Date"]).dt.normalize()

# Clean ticker column one more time to be sure
stocks["Ticker"] = stocks["Ticker"].str.upper().str.replace(".NS", "", regex=False).str.strip()

# Debug: Check data types before merge
print("📊 Stocks Date dtype:", stocks["Date"].dtype)
print("📊 News Date dtype:", news_by_day_ticker["Date"].dtype if not news_by_day_ticker.empty else "N/A (empty)")

# Perform the merge
merged = stocks.merge(news_by_day_ticker, on=["Date", "Ticker"], how="left")

if KEEP_ONLY_MATCHED:
    merged = merged[merged["Headlines"].notna()].copy()

merged.sort_values(["Date", "Ticker"], inplace=True, ignore_index=True)
merged.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("🎯 Final merged:", merged.shape, "→ saved to", OUTPUT_CSV)
print("🎯 Rows with headlines:", (merged["Headlines"].notna()).sum())

# Show some examples of matched data
if not news_by_day_ticker.empty:
    print("\n📰 Sample matched headlines:")
    print(news_by_day_ticker.head(3))
else:
    print("\n⚠️ No headlines were matched! Check the debugging info above.")

print("\nPreview of final data:")
print(merged.head(5))

In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file
df = pd.read_csv('merged_nifty50_company_headlines.csv', sep="\t")

print(f"Initial data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Drop rows without headlines
print(f"\nRows before removing empty headlines: {len(df)}")
df = df.dropna(subset=['Headlines'])
df = df[df['Headlines'].str.strip() != '']  # Remove empty strings
print(f"Rows after removing empty headlines: {len(df)}")

# Convert Date column to datetime (handling DD-MM-YYYY format)
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Calculate additional metrics
df['Daily_Return'] = ((df['Close'] - df['Open']) / df['Open']) * 100
df['Price_Range'] = df['High'] - df['Low']
df['Price_Range_Pct'] = (df['Price_Range'] / df['Open']) * 100
df['Volume_M'] = df['Volume'] / 1000000  # Volume in millions

# Add day of week and month columns for later analysis
df['DayOfWeek'] = df['Date'].dt.day_name()
df['Month'] = df['Date'].dt.to_period('M')

# Categorize headlines count
df['Headlines_Category'] = pd.cut(df['HeadlinesCount'],
                                bins=[0, 1, 3, 5, float('inf')],
                                labels=['Low (1)', 'Medium (2-3)', 'High (4-5)', 'Very High (6+)'])

print("\n" + "="*60)
print("CLEANED DATASET OVERVIEW")
print("="*60)

print(f"Final dataset shape: {df.shape}")
print(f"Date range: {df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"Number of unique tickers: {df['Ticker'].nunique()}")
print(f"Unique tickers: {sorted(df['Ticker'].unique())}")

print("\nMissing values:")
print(df.isnull().sum())

print("\nBasic statistics:")
print(df[['Open', 'High', 'Low', 'Close', 'Volume', 'HeadlinesCount', 'Daily_Return']].describe())

# Save cleaned data
df.to_csv('cleaned_nifty50_data.csv', index=False)
print(f"\nCleaned data saved as 'cleaned_nifty50_data.csv'")
print("Data cleaning completed!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load cleaned data
print("Loading cleaned data...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Creating distribution visualizations...")

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Data Distribution Analysis', fontsize=16, fontweight='bold')

# Daily Returns Distribution
axes[0, 0].hist(df['Daily_Return'], bins=50, alpha=0.7, color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Daily Returns (%)')
axes[0, 0].set_xlabel('Daily Return (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(df['Daily_Return'].mean(), color='red', linestyle='--',
                   label=f'Mean: {df["Daily_Return"].mean():.2f}%')
axes[0, 0].legend()

# Volume Distribution
axes[0, 1].hist(df['Volume_M'], bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribution of Trading Volume (Millions)')
axes[0, 1].set_xlabel('Volume (Millions)')
axes[0, 1].set_ylabel('Frequency')

# Headlines Count Distribution
axes[0, 2].hist(df['HeadlinesCount'], bins=30, alpha=0.7, color='orange', edgecolor='black')
axes[0, 2].set_title('Distribution of Headlines Count')
axes[0, 2].set_xlabel('Headlines Count')
axes[0, 2].set_ylabel('Frequency')

# Price Range Percentage
axes[1, 0].hist(df['Price_Range_Pct'], bins=50, alpha=0.7, color='purple', edgecolor='black')
axes[1, 0].set_title('Distribution of Daily Price Range (%)')
axes[1, 0].set_xlabel('Price Range (%)')
axes[1, 0].set_ylabel('Frequency')

# Top 10 Most Active Stocks by Volume
top_volume_stocks = df.groupby('Ticker')['Volume_M'].mean().nlargest(10)
axes[1, 1].bar(range(len(top_volume_stocks)), top_volume_stocks.values, color='coral')
axes[1, 1].set_title('Top 10 Stocks by Average Volume')
axes[1, 1].set_xlabel('Stocks')
axes[1, 1].set_ylabel('Average Volume (Millions)')
axes[1, 1].set_xticks(range(len(top_volume_stocks)))
axes[1, 1].set_xticklabels(top_volume_stocks.index, rotation=45)

# Headlines vs Returns Scatter
axes[1, 2].scatter(df['HeadlinesCount'], df['Daily_Return'], alpha=0.5, color='teal')
axes[1, 2].set_title('Headlines Count vs Daily Returns')
axes[1, 2].set_xlabel('Headlines Count')
axes[1, 2].set_ylabel('Daily Return (%)')

plt.tight_layout()
plt.savefig('distribution_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Distribution analysis completed and saved as 'distribution_analysis.png'!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load cleaned data
print("Loading cleaned data...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.to_period('M')

print("Performing time series analysis...")

# Monthly analysis
monthly_data = df.groupby('Month').agg({
    'Daily_Return': 'mean',
    'Volume_M': 'mean',
    'HeadlinesCount': 'mean',
    'Close': 'mean'
}).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Time Series Analysis', fontsize=16, fontweight='bold')

# Monthly Average Returns
axes[0, 0].plot(range(len(monthly_data)), monthly_data['Daily_Return'],
                marker='o', linewidth=2, markersize=6)
axes[0, 0].set_title('Monthly Average Daily Returns')
axes[0, 0].set_ylabel('Average Daily Return (%)')
# Show only every 6th month label to avoid crowding
step = max(1, len(monthly_data) // 10)
axes[0, 0].set_xticks(range(0, len(monthly_data), step))
axes[0, 0].set_xticklabels([str(monthly_data.iloc[i]['Month']) for i in range(0, len(monthly_data), step)], rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Monthly Average Volume
axes[0, 1].plot(range(len(monthly_data)), monthly_data['Volume_M'],
                marker='s', color='green', linewidth=2, markersize=6)
axes[0, 1].set_title('Monthly Average Trading Volume')
axes[0, 1].set_ylabel('Average Volume (Millions)')
axes[0, 1].set_xticks(range(0, len(monthly_data), step))
axes[0, 1].set_xticklabels([str(monthly_data.iloc[i]['Month']) for i in range(0, len(monthly_data), step)], rotation=45)
axes[0, 1].grid(True, alpha=0.3)

# Monthly Average Headlines Count
axes[1, 0].plot(range(len(monthly_data)), monthly_data['HeadlinesCount'],
                marker='^', color='orange', linewidth=2, markersize=6)
axes[1, 0].set_title('Monthly Average Headlines Count')
axes[1, 0].set_ylabel('Average Headlines Count')
axes[1, 0].set_xticks(range(0, len(monthly_data), step))
axes[1, 0].set_xticklabels([str(monthly_data.iloc[i]['Month']) for i in range(0, len(monthly_data), step)], rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# Day of Week Analysis
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_returns = df.groupby('DayOfWeek')['Daily_Return'].mean().reindex(day_order)

axes[1, 1].bar(day_returns.index, day_returns.values, color='skyblue', alpha=0.8)
axes[1, 1].set_title('Average Daily Returns by Day of Week')
axes[1, 1].set_ylabel('Average Daily Return (%)')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('time_series_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTime Series Insights:")
print(f"Best performing month: {monthly_data.loc[monthly_data['Daily_Return'].idxmax(), 'Month']}")
print(f"Worst performing month: {monthly_data.loc[monthly_data['Daily_Return'].idxmin(), 'Month']}")
print(f"Highest volume month: {monthly_data.loc[monthly_data['Volume_M'].idxmax(), 'Month']}")
print(f"Most news active month: {monthly_data.loc[monthly_data['HeadlinesCount'].idxmax(), 'Month']}")

print(f"\nBest performing day: {day_returns.idxmax()}")
print(f"Worst performing day: {day_returns.idxmin()}")

print("\nTime series analysis completed and saved as 'time_series_analysis.png'!")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Load cleaned data
print("Loading cleaned data...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

print("Analyzing individual stock performance...")

# Calculate metrics by ticker
stock_metrics = df.groupby('Ticker').agg({
    'Daily_Return': ['mean', 'std'],
    'Volume_M': 'mean',
    'HeadlinesCount': 'mean',
    'Close': ['mean', 'max', 'min'],
    'Price_Range_Pct': 'mean'
}).round(3)

# Flatten column names
stock_metrics.columns = ['_'.join(col).strip() for col in stock_metrics.columns]
stock_metrics = stock_metrics.reset_index()

# Sort by average daily return
stock_metrics = stock_metrics.sort_values('Daily_Return_mean', ascending=False)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Stock Performance Analysis', fontsize=16, fontweight='bold')

# Top 10 Best Performing Stocks
top_10_stocks = stock_metrics.head(10)
axes[0, 0].barh(top_10_stocks['Ticker'], top_10_stocks['Daily_Return_mean'],
                color='green', alpha=0.7)
axes[0, 0].set_title('Top 10 Best Performing Stocks (Avg Daily Return)')
axes[0, 0].set_xlabel('Average Daily Return (%)')

# Bottom 10 Performing Stocks
bottom_10_stocks = stock_metrics.tail(10)
axes[0, 1].barh(bottom_10_stocks['Ticker'], bottom_10_stocks['Daily_Return_mean'],
                color='red', alpha=0.7)
axes[0, 1].set_title('Bottom 10 Performing Stocks (Avg Daily Return)')
axes[0, 1].set_xlabel('Average Daily Return (%)')

# Volatility vs Return (Risk-Return Analysis)
scatter = axes[1, 0].scatter(stock_metrics['Daily_Return_std'], stock_metrics['Daily_Return_mean'],
                            s=stock_metrics['Volume_M_mean']*2, alpha=0.6, c=stock_metrics['HeadlinesCount_mean'],
                            cmap='viridis')
axes[1, 0].set_title('Risk vs Return Analysis\n(Bubble size = Volume, Color = Headlines)')
axes[1, 0].set_xlabel('Volatility (Std Dev of Returns)')
axes[1, 0].set_ylabel('Average Daily Return (%)')
axes[1, 0].grid(True, alpha=0.3)
plt.colorbar(scatter, ax=axes[1, 0], label='Avg Headlines Count')

# Headlines vs Performance
axes[1, 1].scatter(stock_metrics['HeadlinesCount_mean'], stock_metrics['Daily_Return_mean'],
                  alpha=0.7, color='orange', s=60)
axes[1, 1].set_title('Headlines Count vs Performance')
axes[1, 1].set_xlabel('Average Headlines Count')
axes[1, 1].set_ylabel('Average Daily Return (%)')
axes[1, 1].grid(True, alpha=0.3)

# Add trend line for headlines vs performance
z = np.polyfit(stock_metrics['HeadlinesCount_mean'], stock_metrics['Daily_Return_mean'], 1)
p = np.poly1d(z)
axes[1, 1].plot(stock_metrics['HeadlinesCount_mean'], p(stock_metrics['HeadlinesCount_mean']),
                "r--", alpha=0.8, linewidth=2)

plt.tight_layout()
plt.savefig('stock_performance_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Save stock metrics
stock_metrics.to_csv('stock_performance_metrics.csv', index=False)

print("\nTop 5 Best Performing Stocks:")
print(stock_metrics[['Ticker', 'Daily_Return_mean', 'Daily_Return_std', 'Volume_M_mean']].head())

print("\nTop 5 Worst Performing Stocks:")
print(stock_metrics[['Ticker', 'Daily_Return_mean', 'Daily_Return_std', 'Volume_M_mean']].tail())

print("\nMost Volatile Stocks (Top 5):")
most_volatile = stock_metrics.nlargest(5, 'Daily_Return_std')
print(most_volatile[['Ticker', 'Daily_Return_mean', 'Daily_Return_std']])

print("\nMost News-Active Stocks (Top 5):")
most_news_active = stock_metrics.nlargest(5, 'HeadlinesCount_mean')
print(most_news_active[['Ticker', 'HeadlinesCount_mean', 'Daily_Return_mean']])

print("\nStock performance analysis completed!")
print("Results saved as 'stock_performance_analysis.png' and 'stock_performance_metrics.csv'")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')

# Load cleaned data
print("Loading cleaned data...")
df = pd.read_csv('cleaned_nifty50_data.csv')

print("Performing correlation analysis...")

# Select numerical columns for correlation
corr_columns = ['Open', 'High', 'Low', 'Close', 'Volume', 'HeadlinesCount',
                'Daily_Return', 'Price_Range_Pct', 'Volume_M']
correlation_matrix = df[corr_columns].corr()

# Create correlation heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Correlation Analysis', fontsize=16, fontweight='bold')

# Full correlation matrix
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": .8}, ax=axes[0])
axes[0].set_title('Complete Correlation Matrix')

# Focus on correlations with Daily_Return and HeadlinesCount
focus_columns = ['Daily_Return', 'HeadlinesCount', 'Volume_M', 'Price_Range_Pct']
focus_corr = df[focus_columns].corr()
sns.heatmap(focus_corr, annot=True, cmap='RdYlBu_r', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": .8}, ax=axes[1])
axes[1].set_title('Key Metrics Correlation')

plt.tight_layout()
plt.savefig('correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# Print key correlations
print("\nKey Correlation Insights:")
print(f"Headlines Count vs Daily Return: {df['HeadlinesCount'].corr(df['Daily_Return']):.4f}")
print(f"Volume vs Daily Return: {df['Volume_M'].corr(df['Daily_Return']):.4f}")
print(f"Price Range % vs Daily Return: {df['Price_Range_Pct'].corr(df['Daily_Return']):.4f}")
print(f"Headlines Count vs Volume: {df['HeadlinesCount'].corr(df['Volume_M']):.4f}")

# Create scatter plots for key relationships
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Key Relationship Scatter Plots', fontsize=16, fontweight='bold')

# Headlines vs Returns
axes[0, 0].scatter(df['HeadlinesCount'], df['Daily_Return'], alpha=0.5, color='blue')
axes[0, 0].set_xlabel('Headlines Count')
axes[0, 0].set_ylabel('Daily Return (%)')
axes[0, 0].set_title('Headlines Count vs Daily Returns')
axes[0, 0].grid(True, alpha=0.3)

# Add trend line
z = np.polyfit(df['HeadlinesCount'], df['Daily_Return'], 1)
p = np.poly1d(z)
axes[0, 0].plot(df['HeadlinesCount'], p(df['HeadlinesCount']), "r--", alpha=0.8)

# Volume vs Returns
axes[0, 1].scatter(df['Volume_M'], df['Daily_Return'], alpha=0.5, color='green')
axes[0, 1].set_xlabel('Volume (Millions)')
axes[0, 1].set_ylabel('Daily Return (%)')
axes[0, 1].set_title('Volume vs Daily Returns')
axes[0, 1].grid(True, alpha=0.3)

# Price Range vs Returns
axes[1, 0].scatter(df['Price_Range_Pct'], df['Daily_Return'], alpha=0.5, color='red')
axes[1, 0].set_xlabel('Price Range (%)')
axes[1, 0].set_ylabel('Daily Return (%)')
axes[1, 0].set_title('Price Volatility vs Daily Returns')
axes[1, 0].grid(True, alpha=0.3)

# Headlines vs Volume
axes[1, 1].scatter(df['HeadlinesCount'], df['Volume_M'], alpha=0.5, color='purple')
axes[1, 1].set_xlabel('Headlines Count')
axes[1, 1].set_ylabel('Volume (Millions)')
axes[1, 1].set_title('Headlines Count vs Trading Volume')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('correlation_scatter_plots.png', dpi=300, bbox_inches='tight')
plt.show()

# Save correlation matrix
correlation_matrix.to_csv('correlation_matrix.csv')

print("\nCorrelation analysis completed!")
print("Results saved as 'correlation_analysis.png', 'correlation_scatter_plots.png', and 'correlation_matrix.csv'")

# Identify strongest correlations (excluding self-correlations)
strong_correlations = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.5:  # Strong correlation threshold
            strong_correlations.append({
                'Variable 1': correlation_matrix.columns[i],
                'Variable 2': correlation_matrix.columns[j],
                'Correlation': corr_val
            })

if strong_correlations:
    print(f"\nStrong Correlations (|r| > 0.5):")
    for corr in sorted(strong_correlations, key=lambda x: abs(x['Correlation']), reverse=True):
        print(f"{corr['Variable 1']} - {corr['Variable 2']}: {corr['Correlation']:.4f}")
else:
    print("\nNo strong correlations found (|r| > 0.5)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import pearsonr, spearmanr
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("Set2")

# Load cleaned data
print("Loading cleaned data for news impact analysis...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

print(f"Analyzing {df['Ticker'].nunique()} stocks over {len(df)} trading days")
print("="*80)

def calculate_news_impact_metrics():
    """Calculate various metrics to measure news impact on each stock"""

    results = []

    for ticker in df['Ticker'].unique():
        stock_data = df[df['Ticker'] == ticker].copy()

        if len(stock_data) < 10:  # Skip stocks with insufficient data
            continue

        # 1. Correlation Analysis
        corr_return_news, p_val_return = pearsonr(stock_data['HeadlinesCount'], stock_data['Daily_Return'])
        corr_vol_news, p_val_vol = pearsonr(stock_data['HeadlinesCount'], stock_data['Price_Range_Pct'])
        corr_volume_news, p_val_volume = pearsonr(stock_data['HeadlinesCount'], stock_data['Volume_M'])

        # 2. High vs Low News Days Analysis
        median_news = stock_data['HeadlinesCount'].median()
        high_news = stock_data[stock_data['HeadlinesCount'] > median_news]
        low_news = stock_data[stock_data['HeadlinesCount'] <= median_news]

        if len(high_news) > 0 and len(low_news) > 0:
            # T-test for returns
            t_stat_return, t_p_return = stats.ttest_ind(high_news['Daily_Return'], low_news['Daily_Return'])

            # T-test for volatility
            t_stat_vol, t_p_vol = stats.ttest_ind(high_news['Price_Range_Pct'], low_news['Price_Range_Pct'])

            # Effect sizes (mean differences)
            return_effect = high_news['Daily_Return'].mean() - low_news['Daily_Return'].mean()
            vol_effect = high_news['Price_Range_Pct'].mean() - low_news['Price_Range_Pct'].mean()
            volume_effect = high_news['Volume_M'].mean() - low_news['Volume_M'].mean()
        else:
            t_stat_return = t_p_return = t_stat_vol = t_p_vol = np.nan
            return_effect = vol_effect = volume_effect = np.nan

        # 3. News Sensitivity Score (composite metric)
        # Combine correlation strength and statistical significance
        news_sensitivity = (
            abs(corr_return_news) * (1 if p_val_return < 0.05 else 0.5) +
            abs(corr_vol_news) * (1 if p_val_vol < 0.05 else 0.5) +
            abs(corr_volume_news) * (1 if p_val_volume < 0.05 else 0.5)
        ) / 3

        results.append({
            'Ticker': ticker,
            'Sample_Size': len(stock_data),
            'Avg_Headlines': stock_data['HeadlinesCount'].mean(),
            'Avg_Return': stock_data['Daily_Return'].mean(),
            'Avg_Volatility': stock_data['Price_Range_Pct'].mean(),

            # Correlation metrics
            'Return_News_Corr': corr_return_news,
            'Return_News_PValue': p_val_return,
            'Vol_News_Corr': corr_vol_news,
            'Vol_News_PValue': p_val_vol,
            'Volume_News_Corr': corr_volume_news,
            'Volume_News_PValue': p_val_volume,

            # T-test results
            'Return_TTest_Stat': t_stat_return,
            'Return_TTest_PValue': t_p_return,
            'Vol_TTest_Stat': t_stat_vol,
            'Vol_TTest_PValue': t_p_vol,

            # Effect sizes
            'Return_Effect_Size': return_effect,
            'Volatility_Effect_Size': vol_effect,
            'Volume_Effect_Size': volume_effect,

            # Overall news sensitivity
            'News_Sensitivity_Score': news_sensitivity
        })

    return pd.DataFrame(results)

# Calculate news impact metrics
print("Calculating news impact metrics for each stock...")
news_impact_df = calculate_news_impact_metrics()

# Sort by news sensitivity
news_impact_df = news_impact_df.sort_values('News_Sensitivity_Score', ascending=False)

print("Analysis completed! Creating visualizations...")

# Create comprehensive visualizations with better spacing
# Split into multiple figures for better clarity

# FIGURE 1: News Sensitivity Ranking
plt.figure(figsize=(16, 10))
colors = ['red' if score > 0.3 else 'orange' if score > 0.15 else 'green'
          for score in news_impact_df['News_Sensitivity_Score']]
bars = plt.barh(range(len(news_impact_df)), news_impact_df['News_Sensitivity_Score'],
                color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
plt.yticks(range(len(news_impact_df)), news_impact_df['Ticker'], fontsize=10)
plt.xlabel('News Sensitivity Score (Higher = More Affected by News)', fontsize=12)
plt.title('Stock Ranking by News Sensitivity\n(Red: High Impact, Orange: Medium Impact, Green: Low Impact)',
          fontsize=16, fontweight='bold', pad=20)
plt.grid(True, alpha=0.3, axis='x')

# Add score labels
for i, (bar, score) in enumerate(zip(bars, news_impact_df['News_Sensitivity_Score'])):
    plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
             f'{score:.3f}', va='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('news_sensitivity_ranking.png', dpi=300, bbox_inches='tight')
plt.show()

# FIGURE 2: Correlation Analysis (3 subplots)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('News Impact Correlation Analysis', fontsize=16, fontweight='bold', y=1.05)

# Return vs News Correlation
significant = news_impact_df['Return_News_PValue'] < 0.05
colors_return = ['red' if sig else 'lightblue' for sig in significant]
axes[0].scatter(news_impact_df['Return_News_Corr'], range(len(news_impact_df)),
                c=colors_return, alpha=0.7, s=80, edgecolor='black', linewidth=0.5)
axes[0].axvline(x=0, color='black', linestyle='--', alpha=0.7, linewidth=2)
axes[0].set_xlabel('Correlation (Headlines vs Returns)', fontsize=11)
axes[0].set_ylabel('Stocks (Ranked by Sensitivity)', fontsize=11)
axes[0].set_title('Return-News Correlation\n(Red: Significant, Blue: Not Significant)', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Volatility vs News Correlation
significant_vol = news_impact_df['Vol_News_PValue'] < 0.05
colors_vol = ['red' if sig else 'lightblue' for sig in significant_vol]
axes[1].scatter(news_impact_df['Vol_News_Corr'], range(len(news_impact_df)),
                c=colors_vol, alpha=0.7, s=80, edgecolor='black', linewidth=0.5)
axes[1].axvline(x=0, color='black', linestyle='--', alpha=0.7, linewidth=2)
axes[1].set_xlabel('Correlation (Headlines vs Volatility)', fontsize=11)
axes[1].set_ylabel('Stocks (Ranked by Sensitivity)', fontsize=11)
axes[1].set_title('Volatility-News Correlation\n(Red: Significant, Blue: Not Significant)', fontsize=12)
axes[1].grid(True, alpha=0.3)

# Volume vs News Correlation
significant_volume = news_impact_df['Volume_News_PValue'] < 0.05
colors_volume = ['red' if sig else 'lightblue' for sig in significant_volume]
axes[2].scatter(news_impact_df['Volume_News_Corr'], range(len(news_impact_df)),
                c=colors_volume, alpha=0.7, s=80, edgecolor='black', linewidth=0.5)
axes[2].axvline(x=0, color='black', linestyle='--', alpha=0.7, linewidth=2)
axes[2].set_xlabel('Correlation (Headlines vs Volume)', fontsize=11)
axes[2].set_ylabel('Stocks (Ranked by Sensitivity)', fontsize=11)
axes[2].set_title('Volume-News Correlation\n(Red: Significant, Blue: Not Significant)', fontsize=12)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('news_correlation_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# FIGURE 3: Effect Sizes Analysis
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle('Effect Size Analysis: High News Days vs Low News Days', fontsize=16, fontweight='bold', y=1.02)

# Returns Effect Size
significant_effect = news_impact_df['Return_TTest_PValue'] < 0.05
colors_effect = ['darkgreen' if eff > 0 and sig else 'darkred' if eff < 0 and sig else 'gray'
                 for eff, sig in zip(news_impact_df['Return_Effect_Size'], significant_effect)]
bars1 = axes[0].barh(range(len(news_impact_df)), news_impact_df['Return_Effect_Size'],
                     color=colors_effect, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[0].axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
axes[0].set_xlabel('Effect Size (%)', fontsize=11)
axes[0].set_ylabel('Stocks (Ranked by Sensitivity)', fontsize=11)
axes[0].set_title('Return Impact: High News vs Low News Days\n(Green: Positive Impact, Red: Negative Impact)', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='x')

# Volatility Effect Size
significant_vol_effect = news_impact_df['Vol_TTest_PValue'] < 0.05
colors_vol_effect = ['orange' if eff > 0 and sig else 'blue' if eff < 0 and sig else 'gray'
                     for eff, sig in zip(news_impact_df['Volatility_Effect_Size'], significant_vol_effect)]
bars2 = axes[1].barh(range(len(news_impact_df)), news_impact_df['Volatility_Effect_Size'],
                     color=colors_vol_effect, alpha=0.7, edgecolor='black', linewidth=0.5)
axes[1].axvline(x=0, color='black', linestyle='-', alpha=0.8, linewidth=2)
axes[1].set_xlabel('Effect Size (%)', fontsize=11)
axes[1].set_ylabel('Stocks (Ranked by Sensitivity)', fontsize=11)
axes[1].set_title('Volatility Impact: High News vs Low News Days\n(Orange: Increases Volatility)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('news_effect_size_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

# FIGURE 4: Summary Statistics and Distribution
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Summary Statistics and Distribution', fontsize=16, fontweight='bold', y=1.05)

# Summary Statistics
summary_stats = [
    len(news_impact_df[news_impact_df['Return_News_PValue'] < 0.05]),
    len(news_impact_df[news_impact_df['Vol_News_PValue'] < 0.05]),
    len(news_impact_df[news_impact_df['Volume_News_PValue'] < 0.05]),
    len(news_impact_df[news_impact_df['News_Sensitivity_Score'] > 0.3])
]
labels = ['Return\nImpact', 'Volatility\nImpact', 'Volume\nImpact', 'High\nSensitivity']
colors_summary = ['lightcoral', 'lightsalmon', 'lightblue', 'lightgreen']

bars = axes[0].bar(labels, summary_stats, color=colors_summary, alpha=0.8,
                   edgecolor='black', linewidth=1)
axes[0].set_ylabel('Number of Stocks', fontsize=11)
axes[0].set_title('Summary: Stocks Significantly Affected by News', fontsize=12)
axes[0].grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, value in zip(bars, summary_stats):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(value), ha='center', fontweight='bold', fontsize=10)

# News Sensitivity Distribution
axes[1].hist(news_impact_df['News_Sensitivity_Score'], bins=15, alpha=0.7,
             color='skyblue', edgecolor='navy', linewidth=1)
axes[1].axvline(news_impact_df['News_Sensitivity_Score'].mean(), color='red', linestyle='--', linewidth=2,
                label=f'Mean: {news_impact_df["News_Sensitivity_Score"].mean():.3f}')
axes[1].axvline(news_impact_df['News_Sensitivity_Score'].median(), color='orange', linestyle='--', linewidth=2,
                label=f'Median: {news_impact_df["News_Sensitivity_Score"].median():.3f}')
axes[1].set_xlabel('News Sensitivity Score', fontsize=11)
axes[1].set_ylabel('Number of Stocks', fontsize=11)
axes[1].set_title('Distribution of News Sensitivity Scores', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('news_summary_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

# FIGURE 5: Top and Bottom Performers Table (Fixed)
fig, ax = plt.subplots(figsize=(12, 8))
ax.axis('off')

# Create simple, reliable table data
top_5 = news_impact_df.head(5)
bottom_5 = news_impact_df.tail(5)

# Prepare table data - keep it simple
table_data = []
headers = ['Rank', 'Stock', 'Score', 'Impact Level']

# Add top 5 stocks
for i, (_, row) in enumerate(top_5.iterrows(), 1):
    impact_level = 'HIGH' if row['News_Sensitivity_Score'] > 0.3 else 'MEDIUM' if row['News_Sensitivity_Score'] > 0.15 else 'LOW'
    table_data.append([str(i), row['Ticker'], f"{row['News_Sensitivity_Score']:.3f}", impact_level])

# Create first table for TOP 5
table1 = ax.table(cellText=table_data,
                  colLabels=headers,
                  cellLoc='center',
                  loc='upper center',
                  bbox=[0.1, 0.55, 0.8, 0.4])

table1.auto_set_font_size(False)
table1.set_fontsize(11)
table1.scale(1, 2)

# Style the TOP 5 table
# Header row
for j in range(4):
    table1[(0, j)].set_facecolor('#2E4057')
    table1[(0, j)].set_text_props(weight='bold', color='white')

# Data rows
for i in range(1, 6):  # rows 1-5 (data rows)
    for j in range(4):
        if i <= 3:  # Top 3 - darker red
            table1[(i, j)].set_facecolor('#ffcccc')
        else:  # Next 2 - lighter red
            table1[(i, j)].set_facecolor('#ffe6e6')
        table1[(i, j)].set_text_props(weight='bold')

# Add title for top table
ax.text(0.5, 0.98, '🏆 TOP 5 MOST NEWS-SENSITIVE STOCKS',
        ha='center', va='top', fontsize=14, fontweight='bold',
        transform=ax.transAxes)

# Prepare bottom 5 table data
table_data_bottom = []
for i, (_, row) in enumerate(bottom_5.iterrows(), 1):
    impact_level = 'HIGH' if row['News_Sensitivity_Score'] > 0.3 else 'MEDIUM' if row['News_Sensitivity_Score'] > 0.15 else 'LOW'
    table_data_bottom.append([str(i), row['Ticker'], f"{row['News_Sensitivity_Score']:.3f}", impact_level])

# Create second table for BOTTOM 5
table2 = ax.table(cellText=table_data_bottom,
                  colLabels=headers,
                  cellLoc='center',
                  loc='lower center',
                  bbox=[0.1, 0.05, 0.8, 0.4])

table2.auto_set_font_size(False)
table2.set_fontsize(11)
table2.scale(1, 2)

# Style the BOTTOM 5 table
# Header row
for j in range(4):
    table2[(0, j)].set_facecolor('#2E4057')
    table2[(0, j)].set_text_props(weight='bold', color='white')

# Data rows
for i in range(1, 6):  # rows 1-5 (data rows)
    for j in range(4):
        table2[(i, j)].set_facecolor('#ccffcc')  # Light green
        table2[(i, j)].set_text_props(weight='bold')

# Add title for bottom table
ax.text(0.5, 0.48, '🛡️ BOTTOM 5 LEAST NEWS-SENSITIVE STOCKS',
        ha='center', va='top', fontsize=14, fontweight='bold',
        transform=ax.transAxes)

plt.title('News Impact Rankings - Top and Bottom Performers', fontsize=16, fontweight='bold', pad=20)
plt.savefig('news_impact_rankings_table.png', dpi=300, bbox_inches='tight')
plt.show()

# Generate detailed report
print("\n" + "="*80)
print("📰 NEWS IMPACT ANALYSIS REPORT")
print("="*80)

total_stocks = len(news_impact_df)
high_sensitivity = len(news_impact_df[news_impact_df['News_Sensitivity_Score'] > 0.3])
medium_sensitivity = len(news_impact_df[(news_impact_df['News_Sensitivity_Score'] > 0.15) &
                                       (news_impact_df['News_Sensitivity_Score'] <= 0.3)])
low_sensitivity = len(news_impact_df[news_impact_df['News_Sensitivity_Score'] <= 0.15])

print(f"📊 OVERALL SUMMARY:")
print(f"Total Stocks Analyzed: {total_stocks}")
print(f"High News Sensitivity (>0.3): {high_sensitivity} stocks ({high_sensitivity/total_stocks*100:.1f}%)")
print(f"Medium News Sensitivity (0.15-0.3): {medium_sensitivity} stocks ({medium_sensitivity/total_stocks*100:.1f}%)")
print(f"Low News Sensitivity (≤0.15): {low_sensitivity} stocks ({low_sensitivity/total_stocks*100:.1f}%)")

print(f"\n📈 STATISTICAL SIGNIFICANCE:")
return_significant = len(news_impact_df[news_impact_df['Return_News_PValue'] < 0.05])
vol_significant = len(news_impact_df[news_impact_df['Vol_News_PValue'] < 0.05])
volume_significant = len(news_impact_df[news_impact_df['Volume_News_PValue'] < 0.05])

print(f"Stocks with significant return-news correlation: {return_significant}/{total_stocks} ({return_significant/total_stocks*100:.1f}%)")
print(f"Stocks with significant volatility-news correlation: {vol_significant}/{total_stocks} ({vol_significant/total_stocks*100:.1f}%)")
print(f"Stocks with significant volume-news correlation: {volume_significant}/{total_stocks} ({volume_significant/total_stocks*100:.1f}%)")

print(f"\n🏆 TOP 10 MOST NEWS-SENSITIVE STOCKS:")
for i, (_, row) in enumerate(news_impact_df.head(10).iterrows(), 1):
    print(f"{i:2d}. {row['Ticker']} - Score: {row['News_Sensitivity_Score']:.3f}")

print(f"\n🛡️  BOTTOM 5 LEAST NEWS-SENSITIVE STOCKS:")
for i, (_, row) in enumerate(news_impact_df.tail(5).iterrows(), 1):
    print(f"{i}. {row['Ticker']} - Score: {row['News_Sensitivity_Score']:.3f}")

# Save detailed results
news_impact_df.to_csv('detailed_news_impact_analysis.csv', index=False)

print(f"\n💾 RESULTS SAVED:")
print("- News Sensitivity Ranking: 'news_sensitivity_ranking.png'")
print("- Correlation Analysis: 'news_correlation_analysis.png'")
print("- Effect Size Analysis: 'news_effect_size_analysis.png'")
print("- Summary & Distribution: 'news_summary_distribution.png'")
print("- Rankings Table: 'news_impact_rankings_table.png'")
print("- Detailed Data: 'detailed_news_impact_analysis.csv'")

print(f"\n🔍 KEY INSIGHTS:")
avg_sensitivity = news_impact_df['News_Sensitivity_Score'].mean()
if avg_sensitivity > 0.25:
    print("• NIFTY 50 stocks show HIGH overall sensitivity to news")
elif avg_sensitivity > 0.15:
    print("• NIFTY 50 stocks show MODERATE overall sensitivity to news")
else:
    print("• NIFTY 50 stocks show LOW overall sensitivity to news")

pos_correlation = len(news_impact_df[news_impact_df['Return_News_Corr'] > 0])
neg_correlation = len(news_impact_df[news_impact_df['Return_News_Corr'] < 0])

if pos_correlation > neg_correlation:
    print("• More stocks tend to have POSITIVE returns with increased news")
else:
    print("• More stocks tend to have NEGATIVE returns with increased news")

print("\n" + "="*80)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
print("Loading data for REALISTIC news impact analysis...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

print(f"Total dataset: {len(df)} records")

def calculate_market_realistic_thresholds(daily_data):
    """Calculate thresholds based on actual market behavior and trading reality"""

    # Calculate market statistics
    returns = daily_data['Daily_Return'].abs()

    print("📊 MARKET STATISTICS ANALYSIS:")
    print("="*50)
    print(f"Average daily absolute return: {returns.mean():.3f}%")
    print(f"Standard deviation: {returns.std():.3f}%")
    print(f"Median daily move: {returns.median():.3f}%")
    print(f"75th percentile: {returns.quantile(0.75):.3f}%")
    print(f"90th percentile: {returns.quantile(0.90):.3f}%")
    print(f"95th percentile: {returns.quantile(0.95):.3f}%")
    print(f"99th percentile: {returns.quantile(0.99):.3f}%")

    # Real-world trading thresholds based on market behavior
    thresholds = {
        # Based on actual market impact that traders care about
        'noise_threshold': returns.quantile(0.5),  # Below median = just noise
        'noticeable_threshold': returns.quantile(0.70),  # 70th percentile = traders notice
        'significant_threshold': returns.quantile(0.85),  # 85th percentile = significant move
        'major_threshold': returns.quantile(0.95),  # 95th percentile = major market event

        # Alternative: Fixed real-world thresholds
        'intraday_noise': 0.3,      # Less than 0.3% = intraday noise
        'trader_attention': 0.75,   # 0.75% = gets trader attention
        'significant_move': 1.5,    # 1.5% = significant for NIFTY 50
        'major_event': 2.5          # 2.5% = major market event for NIFTY 50
    }

    print(f"\n📈 CALCULATED THRESHOLDS:")
    print(f"Market Noise Level: {thresholds['noise_threshold']:.3f}%")
    print(f"Noticeable Move Level: {thresholds['noticeable_threshold']:.3f}%")
    print(f"Significant Move Level: {thresholds['significant_threshold']:.3f}%")
    print(f"Major Event Level: {thresholds['major_threshold']:.3f}%")

    print(f"\n📊 INDUSTRY STANDARD THRESHOLDS:")
    print(f"Intraday Noise: <{thresholds['intraday_noise']}%")
    print(f"Trader Attention: ≥{thresholds['trader_attention']}%")
    print(f"Significant Move: ≥{thresholds['significant_move']}%")
    print(f"Major Event: ≥{thresholds['major_event']}%")

    return thresholds

def realistic_news_impact_analysis(daily_data):
    """Analyze news impact using realistic, market-based thresholds"""

    thresholds = calculate_market_realistic_thresholds(daily_data)

    # Use the more realistic fixed thresholds based on NIFTY 50 behavior
    categories = {
        'Major Market Impact': 0,      # ≥2.5% - Clear major impact
        'Significant Impact': 0,       # 1.5-2.5% - Definitely impactful
        'Noticeable Impact': 0,        # 0.75-1.5% - Traders notice
        'Minor/Noise Impact': 0,       # 0.3-0.75% - Barely noticeable
        'No Impact': 0,                # <0.3% - Market noise level
        'No News Days': 0              # Days without news
    }

    category_details = {cat: [] for cat in categories.keys()}

    print(f"\n🔍 ANALYZING {len(daily_data)} TRADING DAYS...")

    for idx, row in daily_data.iterrows():
        abs_return = abs(row['Daily_Return'])

        if row['HeadlinesCount'] == 0:
            categories['No News Days'] += 1
            category_details['No News Days'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': 0, 'abs_return': abs_return,
                'description': 'Pure market movement without news'
            })
        elif abs_return >= thresholds['major_event']:  # ≥2.5%
            categories['Major Market Impact'] += 1
            category_details['Major Market Impact'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': int(row['HeadlinesCount']), 'abs_return': abs_return,
                'description': f'Major news-driven {abs_return:.1f}% move'
            })
        elif abs_return >= thresholds['significant_move']:  # 1.5-2.5%
            categories['Significant Impact'] += 1
            category_details['Significant Impact'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': int(row['HeadlinesCount']), 'abs_return': abs_return,
                'description': f'Significant news impact {abs_return:.1f}%'
            })
        elif abs_return >= thresholds['trader_attention']:  # 0.75-1.5%
            categories['Noticeable Impact'] += 1
            category_details['Noticeable Impact'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': int(row['HeadlinesCount']), 'abs_return': abs_return,
                'description': f'Noticeable news effect {abs_return:.1f}%'
            })
        elif abs_return >= thresholds['intraday_noise']:  # 0.3-0.75%
            categories['Minor/Noise Impact'] += 1
            category_details['Minor/Noise Impact'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': int(row['HeadlinesCount']), 'abs_return': abs_return,
                'description': f'Minor/unclear news impact {abs_return:.1f}%'
            })
        else:  # <0.3%
            categories['No Impact'] += 1
            category_details['No Impact'].append({
                'date': row['Date'], 'return': row['Daily_Return'],
                'headlines': int(row['HeadlinesCount']), 'abs_return': abs_return,
                'description': f'News present but no market impact {abs_return:.1f}%'
            })

    return categories, category_details, thresholds

def create_realistic_pie_charts(categories, category_details, thresholds):
    """Create pie charts based on realistic market thresholds"""

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle('REALISTIC News Impact Analysis: Market-Based Thresholds\nBased on Actual Trading Behavior and NIFTY 50 Characteristics',
                 fontsize=18, fontweight='bold', y=1.02)

    # Chart 1: Realistic News Impact Distribution
    ax1 = axes[0, 0]
    sizes = list(categories.values())
    labels = [f'{key}\n{value} days\n({value/sum(sizes)*100:.1f}%)'
              for key, value in categories.items()]

    impact_colors = ['#8B0000', '#DC143C', '#FF8C00', '#FFD700', '#D3D3D3', '#808080']

    wedges, texts = ax1.pie(
        sizes,
        colors=impact_colors,
        startangle=90,
        autopct=None,
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'},
        labeldistance=1.15,   # push labels outside
        pctdistance=0.85      # percentage distance
    )

    # Add legend instead of crowded labels
    ax1.legend(
        wedges, labels,
        title="Impact Categories",
        loc="center left",
        bbox_to_anchor=(1, 0, 0.5, 1),
        fontsize=9
    )

    ax1.set_title(
        'News Impact Distribution\n(Market-Realistic Thresholds)',
        fontsize=14, fontweight='bold', pad=20
    )


    plt.tight_layout(rect=[0, 0, 1, 0.95])  # <-- leaves space for suptitle
    plt.subplots_adjust(top=0.90)

    # Chart 2: Effective vs Ineffective News (Realistic)
    ax2 = axes[0, 1]
    truly_effective = categories['Major Market Impact'] + categories['Significant Impact']
    somewhat_effective = categories['Noticeable Impact']
    ineffective = categories['Minor/Noise Impact'] + categories['No Impact']
    no_news = categories['No News Days']

    effectiveness_data = [truly_effective, somewhat_effective, ineffective, no_news]
    effectiveness_labels = [
        f'Truly Effective News\n(≥1.5% moves)\n{truly_effective} days\n{truly_effective/sum(effectiveness_data)*100:.1f}%',
        f'Somewhat Effective\n(0.75-1.5% moves)\n{somewhat_effective} days\n{somewhat_effective/sum(effectiveness_data)*100:.1f}%',
        f'Ineffective News\n(<0.75% moves)\n{ineffective} days\n{ineffective/sum(effectiveness_data)*100:.1f}%',
        f'No News\n{no_news} days\n{no_news/sum(effectiveness_data)*100:.1f}%'
    ]

    effectiveness_colors = ['#228B22', '#90EE90', '#FFB6C1', '#D3D3D3']

    ax2.pie(effectiveness_data, labels=effectiveness_labels, colors=effectiveness_colors,
           autopct='', startangle=90, textprops={'fontsize': 9, 'fontweight': 'bold'})
    ax2.set_title('Realistic News Effectiveness\n(Trader-Relevant Thresholds)',
                 fontsize=14, fontweight='bold', pad=20)

    # Chart 3: Market Reality Check
    ax3 = axes[1, 0]
    total_news_days = sum(categories.values()) - categories['No News Days']

    meaningful_impact = truly_effective + somewhat_effective  # ≥0.75%
    meaningless_impact = ineffective  # <0.75%

    reality_data = [meaningful_impact, meaningless_impact]
    reality_labels = [
        f'News with Meaningful Impact\n(≥0.75% moves)\n{meaningful_impact:.0f} days\n{meaningful_impact/total_news_days*100:.1f}% of news days',
        f'News with No Meaningful Impact\n(<0.75% moves)\n{meaningless_impact:.0f} days\n{meaningless_impact/total_news_days*100:.1f}% of news days'
    ]

    ax3.pie(reality_data, labels=reality_labels, colors=['#4CAF50', '#F44336'],
           autopct='%1.1f%%', startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
    ax3.set_title('Market Reality: Do Headlines Matter?\n(0.75% threshold = minimum trader attention)',
                 fontsize=14, fontweight='bold', pad=20)

    # Chart 4: Key Statistics and Thresholds
    ax4 = axes[1, 1]
    ax4.axis('off')

    # Calculate key realistic statistics
    total_days = sum(categories.values())
    news_days = total_days - categories['No News Days']

    # Real success rates
    major_success_rate = categories['Major Market Impact'] / news_days * 100
    significant_success_rate = (categories['Major Market Impact'] + categories['Significant Impact']) / news_days * 100
    meaningful_success_rate = meaningful_impact / news_days * 100

    stats_text = f"""
📊 REALISTIC MARKET ANALYSIS

🎯 THRESHOLDS USED (Market-Based):
• Major Impact: ≥{thresholds['major_event']:.1f}% (Clear market event)
• Significant Impact: ≥{thresholds['significant_move']:.1f}% (Traders act)
• Noticeable Impact: ≥{thresholds['trader_attention']:.1f}% (Traders notice)
• Market Noise: <{thresholds['intraday_noise']:.1f}% (Ignored by traders)

📈 REAL SUCCESS RATES:
• Major Market Events: {major_success_rate:.1f}% of news
• Significant Moves: {significant_success_rate:.1f}% of news
• Meaningful Impact: {meaningful_success_rate:.1f}% of news

🏆 KEY INSIGHTS:
• Only {meaningful_success_rate:.1f}% of news meaningfully moves markets
• {100-meaningful_success_rate:.1f}% of news is market noise
• {categories['No News Days']/total_days*100:.1f}% of trading happens without news

⚡ BOTTOM LINE:
News headlines are effective only {meaningful_success_rate:.0f}% of the time
in creating moves that traders actually care about!
    """

    ax4.text(0.05, 0.95, stats_text, transform=ax4.transAxes, fontsize=11,
            verticalalignment='top', fontweight='bold',
            bbox=dict(boxstyle='round,pad=1', facecolor='lightyellow', alpha=0.9))

    plt.tight_layout()
    plt.subplots_adjust(top=0.92)
    return fig

# Run the realistic analysis
print("="*80)
print("🎯 REALISTIC NEWS IMPACT ANALYSIS")
print("Using Market-Based Thresholds from Actual Trading Behavior")
print("="*80)

# Create daily aggregated data
daily_data = df.groupby('Date').agg({
    'Daily_Return': 'mean',
    'HeadlinesCount': 'sum',
    'Volume_M': 'sum',
    'Price_Range_Pct': 'mean'
}).reset_index()

# Run realistic analysis
categories, category_details, thresholds = realistic_news_impact_analysis(daily_data)

print("\nCreating realistic pie chart visualization...")
fig = create_realistic_pie_charts(categories, category_details, thresholds)

# Save the visualization
plt.savefig('realistic_news_impact_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Print realistic conclusions
print("\n" + "="*80)
print("🔍 REALISTIC MARKET CONCLUSIONS")
print("="*80)

total_days = sum(categories.values())
news_days = total_days - categories['No News Days']

print(f"\n📊 REALISTIC BREAKDOWN:")
for category, count in categories.items():
    percentage = count / total_days * 100
    if category != 'No News Days':
        news_percentage = count / news_days * 100 if news_days > 0 else 0
        print(f"{category}: {count:,} days ({percentage:.1f}% of all days, {news_percentage:.1f}% of news days)")
    else:
        print(f"{category}: {count:,} days ({percentage:.1f}% of all days)")

# The realistic answer
meaningful_impact = categories['Major Market Impact'] + categories['Significant Impact'] + categories['Noticeable Impact']
meaningful_success_rate = meaningful_impact / news_days * 100 if news_days > 0 else 0

truly_effective = categories['Major Market Impact'] + categories['Significant Impact']
true_success_rate = truly_effective / news_days * 100 if news_days > 0 else 0

print(f"\n🎯 THE REALISTIC ANSWER:")
print(f"➤ News headlines meaningfully impact stock prices {meaningful_success_rate:.1f}% of the time")
print(f"➤ News headlines create significant market moves {true_success_rate:.1f}% of the time")
print(f"➤ News headlines fail to move markets {100-meaningful_success_rate:.1f}% of the time")

print(f"\n💡 REAL-WORLD INTERPRETATION:")
if meaningful_success_rate < 30:
    interpretation = "Most news is just noise - markets move on fundamentals and sentiment"
elif meaningful_success_rate < 50:
    interpretation = "News has moderate impact - fundamentals still dominate"
else:
    interpretation = "News significantly drives market movements"

print(f"➤ {interpretation}")

print(f"\n📋 PRACTICAL IMPLICATIONS FOR TRADERS:")
print(f"• Ignore news causing <{thresholds['trader_attention']}% moves (market noise)")
print(f"• Pay attention to news causing ≥{thresholds['significant_move']}% moves")
print(f"• Major opportunities come from ≥{thresholds['major_event']}% news-driven moves")
print(f"• {categories['No News Days']/total_days*100:.1f}% of market moves happen without news")

# Save realistic breakdown
realistic_df = pd.DataFrame([
    {'Category': k, 'Days': v, 'Percentage_All_Days': v/total_days*100,
     'Percentage_News_Days': v/news_days*100 if k != 'No News Days' and news_days > 0 else 0}
    for k, v in categories.items()
])
realistic_df.to_csv('realistic_news_impact_breakdown.csv', index=False)

print(f"\n💾 FILES SAVED:")
print("• Visualization: 'realistic_news_impact_analysis.png'")
print("• Data: 'realistic_news_impact_breakdown.csv'")

print("\n" + "="*80)
print("🎯 FINAL ANSWER: Based on realistic market thresholds,")
print(f"   news headlines effectively impact stock prices {meaningful_success_rate:.0f}% of the time")
print("="*80)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import argrelextrema
import warnings
warnings.filterwarnings('ignore')

# Load data
print("Loading data for Plotly market visualization...")
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# Create daily aggregated data
daily_data = df.groupby('Date').agg({
    'Daily_Return': 'mean',
    'HeadlinesCount': 'sum',
    'Volume_M': 'sum',
    'Price_Range_Pct': 'mean',
    'Close': 'mean'
}).reset_index()

daily_data = daily_data.sort_values('Date').reset_index(drop=True)
daily_data['Year'] = daily_data['Date'].dt.year

available_years = sorted(daily_data['Year'].unique())
print(f"Available years: {available_years}")

def get_event_reason(date, headlines_count, daily_return, volume, avg_volume):
    """Generate detailed reason for market movement"""

    reasons = []

    # Analyze based on news intensity
    if headlines_count >= 20:
        if daily_return > 0:
            reasons.append("Major positive news event/policy announcement")
        else:
            reasons.append("Major negative news/market concerns")
    elif headlines_count >= 12:
        if daily_return > 0:
            reasons.append("Positive sector/corporate developments")
        else:
            reasons.append("Sector concerns/profit booking pressure")
    elif headlines_count >= 5:
        reasons.append("Regular market news flow")
    else:
        if abs(daily_return) > 1.5:
            reasons.append("Technical breakout/breakdown")
        else:
            reasons.append("Sentiment-driven movement")

    # Volume analysis
    if volume > avg_volume * 1.5:
        reasons.append("Heavy institutional participation")
    elif volume > avg_volume * 1.2:
        reasons.append("Increased trading activity")

    # Format output
    main_reason = reasons[0]
    additional = " | ".join(reasons[1:]) if len(reasons) > 1 else ""

    return f"{main_reason}<br>{additional}" if additional else main_reason

def find_all_peaks_valleys(year_data, min_prominence=0.1):
    """Detect ALL visible peaks and valleys including subtle ones"""

    year_data = year_data.sort_values('Date').reset_index(drop=True)

    # Calculate cumulative returns
    year_data['Cumulative_Return'] = (1 + year_data['Daily_Return']/100).cumprod()
    year_data['Cumulative_Return_Pct'] = (year_data['Cumulative_Return'] - 1) * 100

    # Add smoothed version with minimal smoothing to preserve sharp edges
    window = 2  # Reduced from 3 to preserve sharp peaks
    year_data['Cumulative_Smoothed'] = year_data['Cumulative_Return_Pct'].rolling(window, center=True, min_periods=1).mean()

    cumulative = year_data['Cumulative_Smoothed'].values

    events = []
    avg_volume = year_data['Volume_M'].mean()

    # Method 1: Aggressive scipy detection with multiple very small orders
    for order in [1, 2, 3, 4, 5, 6, 7, 8]:  # Added order=1 and 2 for sharp spikes
        peaks = argrelextrema(cumulative, np.greater, order=order)[0]
        valleys = argrelextrema(cumulative, np.less, order=order)[0]

        for peak_idx in peaks:
            if peak_idx >= len(year_data):
                continue
            row = year_data.iloc[peak_idx]

            # Very low threshold to catch all visible peaks
            if abs(row['Cumulative_Return_Pct']) > 0.1 or abs(row['Daily_Return']) > 0.2:
                reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                        row['Daily_Return'], row['Volume_M'], avg_volume)

                # Higher score for sharper moves
                sharpness = abs(row['Daily_Return']) if abs(row['Daily_Return']) > 0.5 else 0

                events.append({
                    'Date': row['Date'],
                    'Cumulative': row['Cumulative_Return_Pct'],
                    'DailyReturn': row['Daily_Return'],
                    'Type': 'Peak',
                    'Headlines': int(row['HeadlinesCount']),
                    'Reason': reason,
                    'Score': abs(row['Cumulative_Return_Pct']) * 0.5 + abs(row['Daily_Return']) * 2 + sharpness * 3,
                    'Method': f'scipy_order{order}'
                })

        for valley_idx in valleys:
            if valley_idx >= len(year_data):
                continue
            row = year_data.iloc[valley_idx]

            if abs(row['Cumulative_Return_Pct']) > 0.1 or abs(row['Daily_Return']) > 0.2:
                reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                        row['Daily_Return'], row['Volume_M'], avg_volume)

                sharpness = abs(row['Daily_Return']) if abs(row['Daily_Return']) > 0.5 else 0

                events.append({
                    'Date': row['Date'],
                    'Cumulative': row['Cumulative_Return_Pct'],
                    'DailyReturn': row['Daily_Return'],
                    'Type': 'Valley',
                    'Headlines': int(row['HeadlinesCount']),
                    'Reason': reason,
                    'Score': abs(row['Cumulative_Return_Pct']) * 0.5 + abs(row['Daily_Return']) * 2 + sharpness * 3,
                    'Method': f'scipy_order{order}'
                })

    # Method 2: Detect ALL significant daily moves (lowered threshold)
    extreme_threshold = np.percentile(np.abs(year_data['Daily_Return']), 75)  # Changed from 85 to 75

    for idx, row in year_data.iterrows():
        if abs(row['Daily_Return']) >= extreme_threshold:
            reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                    row['Daily_Return'], row['Volume_M'], avg_volume)

            events.append({
                'Date': row['Date'],
                'Cumulative': row['Cumulative_Return_Pct'],
                'DailyReturn': row['Daily_Return'],
                'Type': 'Peak' if row['Daily_Return'] > 0 else 'Valley',
                'Headlines': int(row['HeadlinesCount']),
                'Reason': reason,
                'Score': abs(row['Daily_Return']) * 4,
                'Method': 'extreme_day'
            })

    # Method 3: Sharp spike detection - look for sudden changes
    year_data['Price_Velocity'] = year_data['Cumulative_Return_Pct'].diff()
    year_data['Price_Acceleration'] = year_data['Price_Velocity'].diff()

    # Detect sharp spikes based on acceleration
    spike_threshold = np.percentile(np.abs(year_data['Price_Acceleration'].dropna()), 70)

    for i in range(2, len(year_data)):
        if abs(year_data['Price_Acceleration'].iloc[i]) > spike_threshold:
            row = year_data.iloc[i]

            # Check if this is actually a visible turning point
            if i >= 1 and i < len(year_data) - 1:
                prev_val = year_data['Cumulative_Return_Pct'].iloc[i-1]
                curr_val = year_data['Cumulative_Return_Pct'].iloc[i]
                next_val = year_data['Cumulative_Return_Pct'].iloc[i+1]

                is_peak = (curr_val > prev_val and curr_val >= next_val)
                is_valley = (curr_val < prev_val and curr_val <= next_val)

                if is_peak or is_valley:
                    reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                            row['Daily_Return'], row['Volume_M'], avg_volume)

                    events.append({
                        'Date': row['Date'],
                        'Cumulative': row['Cumulative_Return_Pct'],
                        'DailyReturn': row['Daily_Return'],
                        'Type': 'Peak' if is_peak else 'Valley',
                        'Headlines': int(row['HeadlinesCount']),
                        'Reason': reason,
                        'Score': abs(year_data['Price_Acceleration'].iloc[i]) * 5 + abs(row['Daily_Return']) * 2,
                        'Method': 'sharp_spike'
                    })

    # Method 4: Rolling window local extrema (very small windows for sharp edges)
    for window_size in [2, 3, 4, 5]:
        for i in range(window_size, len(year_data) - window_size):
            window_before = year_data['Cumulative_Return_Pct'].iloc[i-window_size:i].values
            window_after = year_data['Cumulative_Return_Pct'].iloc[i+1:i+window_size+1].values
            current = year_data['Cumulative_Return_Pct'].iloc[i]

            # Very sensitive detection for visible peaks/valleys
            is_peak = (current >= window_before.max() and current >= window_after.max())
            is_valley = (current <= window_before.min() and current <= window_after.min())

            if is_peak or is_valley:
                row = year_data.iloc[i]
                reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                        row['Daily_Return'], row['Volume_M'], avg_volume)

                events.append({
                    'Date': row['Date'],
                    'Cumulative': row['Cumulative_Return_Pct'],
                    'DailyReturn': row['Daily_Return'],
                    'Type': 'Peak' if is_peak else 'Valley',
                    'Headlines': int(row['HeadlinesCount']),
                    'Reason': reason,
                    'Score': abs(row['Cumulative_Return_Pct']) * 1 + abs(row['Daily_Return']) * 1.5,
                    'Method': f'rolling_w{window_size}'
                })

    # Method 5: Detect inflection points (change in trend direction)
    year_data['MA_3'] = year_data['Cumulative_Return_Pct'].rolling(3, center=True).mean()
    year_data['MA_5'] = year_data['Cumulative_Return_Pct'].rolling(5, center=True).mean()
    year_data['Trend_3'] = year_data['MA_3'].diff()
    year_data['Trend_5'] = year_data['MA_5'].diff()

    for i in range(5, len(year_data) - 5):
        # Check for trend reversal
        trend_before_3 = year_data['Trend_3'].iloc[i-3:i].mean()
        trend_after_3 = year_data['Trend_3'].iloc[i+1:i+4].mean()

        # Significant trend change
        if (trend_before_3 > 0 and trend_after_3 < 0) or (trend_before_3 < 0 and trend_after_3 > 0):
            row = year_data.iloc[i]
            reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                    row['Daily_Return'], row['Volume_M'], avg_volume)

            is_peak = trend_before_3 > 0 and trend_after_3 < 0

            events.append({
                'Date': row['Date'],
                'Cumulative': row['Cumulative_Return_Pct'],
                'DailyReturn': row['Daily_Return'],
                'Type': 'Peak' if is_peak else 'Valley',
                'Headlines': int(row['HeadlinesCount']),
                'Reason': reason,
                'Score': abs(trend_before_3 - trend_after_3) * 5 + abs(row['Daily_Return']),
                'Method': 'inflection'
            })

    # Method 6: Sliding window maxima/minima (very fine-grained)
    window_sizes = [len(year_data) // 20, len(year_data) // 15, len(year_data) // 10]

    for chunk_size in window_sizes:
        if chunk_size < 5:
            chunk_size = 5

        for i in range(0, len(year_data), max(chunk_size // 3, 1)):  # Overlapping windows
            chunk = year_data.iloc[i:min(i+chunk_size, len(year_data))]

            if len(chunk) < 3:
                continue

            # Find all local max/min in chunk
            chunk_cumulative = chunk['Cumulative_Return_Pct'].values
            local_peaks = argrelextrema(chunk_cumulative, np.greater, order=1)[0]
            local_valleys = argrelextrema(chunk_cumulative, np.less, order=1)[0]

            for peak_idx in local_peaks:
                actual_idx = i + peak_idx
                if actual_idx >= len(year_data):
                    continue

                row = year_data.iloc[actual_idx]
                reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                        row['Daily_Return'], row['Volume_M'], avg_volume)

                events.append({
                    'Date': row['Date'],
                    'Cumulative': row['Cumulative_Return_Pct'],
                    'DailyReturn': row['Daily_Return'],
                    'Type': 'Peak',
                    'Headlines': int(row['HeadlinesCount']),
                    'Reason': reason,
                    'Score': abs(row['Cumulative_Return_Pct']) * 1.2 + abs(row['Daily_Return']) * 1.5,
                    'Method': 'sliding_window'
                })

            for valley_idx in local_valleys:
                actual_idx = i + valley_idx
                if actual_idx >= len(year_data):
                    continue

                row = year_data.iloc[actual_idx]
                reason = get_event_reason(row['Date'], row['HeadlinesCount'],
                                        row['Daily_Return'], row['Volume_M'], avg_volume)

                events.append({
                    'Date': row['Date'],
                    'Cumulative': row['Cumulative_Return_Pct'],
                    'DailyReturn': row['Daily_Return'],
                    'Type': 'Valley',
                    'Headlines': int(row['HeadlinesCount']),
                    'Reason': reason,
                    'Score': abs(row['Cumulative_Return_Pct']) * 1.2 + abs(row['Daily_Return']) * 1.5,
                    'Method': 'sliding_window'
                })

    # Convert to DataFrame and process
    events_df = pd.DataFrame(events)

    if len(events_df) > 0:
        # Remove exact duplicates (same date)
        events_df = events_df.sort_values(['Date', 'Score'], ascending=[True, False])
        events_df = events_df.drop_duplicates(subset=['Date'], keep='first')

        # Sort by date to maintain chronological order
        events_df = events_df.sort_values('Date')

        # Smart filtering: Keep events that are visually distinct
        # Reduced proximity to allow more events, especially in volatile periods
        filtered = []

        for _, event in events_df.iterrows():
            # Much shorter proximity window - only skip if events are within 1-2 days
            # This allows us to mark most visible peaks
            proximity_days = 1

            # Check if this event is too close to existing ones
            too_close = False
            for f in filtered:
                days_apart = abs((f['Date'] - event['Date']).days)
                if days_apart <= proximity_days:
                    # Keep the one with higher score
                    if event['Score'] > f['Score']:
                        filtered.remove(f)
                        break
                    else:
                        too_close = True
                        break

            if not too_close:
                filtered.append(event.to_dict())

        # Sort final results by date
        filtered_df = pd.DataFrame(filtered).sort_values('Date')

        return filtered_df, year_data

    return pd.DataFrame(), year_data

def create_plotly_market_story(year_data, events_df, year):
    """Create interactive Plotly chart with ALL peaks and valleys marked"""

    final_return = year_data['Cumulative_Return_Pct'].iloc[-1]
    best_day = year_data['Daily_Return'].max()
    worst_day = year_data['Daily_Return'].min()
    volatility = year_data['Daily_Return'].std()

    # Create figure
    fig = go.Figure()

    # Add cumulative return line
    line_color = '#2E8B57' if final_return >= 0 else '#DC143C'
    fill_color = 'rgba(46, 139, 87, 0.2)' if final_return >= 0 else 'rgba(220, 20, 60, 0.2)'

    fig.add_trace(go.Scatter(
        x=year_data['Date'],
        y=year_data['Cumulative_Return_Pct'],
        mode='lines',
        name='Cumulative Return',
        line=dict(color=line_color, width=2.5),
        fill='tozeroy',
        fillcolor=fill_color,
        hovertemplate='<b>%{x|%b %d}</b><br>Return: %{y:.2f}%<extra></extra>'
    ))

    # Add markers for ALL detected events
    if len(events_df) > 0:
        for idx, event in events_df.iterrows():
            marker_color = '#228B22' if event['DailyReturn'] > 0 else '#B22222'
            symbol = 'triangle-up' if event['DailyReturn'] > 0 else 'triangle-down'

            # Vary marker size based on significance
            marker_size = min(18, 10 + abs(event['DailyReturn']) * 2)

            # Add marker
            fig.add_trace(go.Scatter(
                x=[event['Date']],
                y=[event['Cumulative']],
                mode='markers',
                marker=dict(
                    size=marker_size,
                    color=marker_color,
                    symbol=symbol,
                    line=dict(color='white', width=1.5)
                ),
                showlegend=False,
                hovertemplate=(
                    f"<b>{event['Date'].strftime('%b %d, %Y')}</b><br>"
                    f"Daily Move: <b>{event['DailyReturn']:+.2f}%</b><br>"
                    f"Cumulative: {event['Cumulative']:.2f}%<br>"
                    f"Headlines: {event['Headlines']}<br>"
                    f"<br><b>Reason:</b><br>{event['Reason']}"
                    "<extra></extra>"
                )
            ))

    # Add zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)

    # Update layout
    fig.update_layout(
        title=dict(
            text=f'<b>NIFTY 50 Market Journey: {year}</b><br>'
                 f'<sub>Total Return: {final_return:.1f}% | Best Day: +{best_day:.1f}% | '
                 f'Worst Day: {worst_day:.1f}% | Volatility: {volatility:.1f}% | Events Detected: {len(events_df)}</sub>',
            x=0.5,
            xanchor='center',
            font=dict(size=18)
        ),
        xaxis_title='Date',
        yaxis_title='Cumulative Return (%)',
        hovermode='closest',
        plot_bgcolor='#FAFAFA',
        paper_bgcolor='white',
        font=dict(family='Arial', size=12),
        height=800,
        width=1600,
        showlegend=False,
        xaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            gridwidth=0.5
        ),
        yaxis=dict(
            showgrid=True,
            gridcolor='lightgray',
            gridwidth=0.5,
            zeroline=True,
            zerolinecolor='gray',
            zerolinewidth=2
        )
    )

    return fig

# Generate Plotly charts for each year
def generate_all_plotly_stories():
    """Generate interactive Plotly charts for all years with ALL peaks/valleys marked"""

    print(f"Generating comprehensive market stories for {len(available_years)} years...")
    print("NOTE: This will detect and mark ALL visible peaks and valleys")

    for year in available_years:
        print(f"\nAnalyzing {year}...")

        year_data = daily_data[daily_data['Year'] == year].copy()

        if len(year_data) < 50:
            print(f"Skipping {year} - insufficient data")
            continue

        # Find ALL peaks and valleys
        events_df, year_data = find_all_peaks_valleys(year_data)

        print(f"Found {len(events_df)} peaks/valleys for {year}")

        # Create Plotly chart
        fig = create_plotly_market_story(year_data, events_df, year)


        # Show in browser/notebook
        fig.show()

        # Print summary statistics
        if len(events_df) > 0:
            peaks = events_df[events_df['Type'] == 'Peak']
            valleys = events_df[events_df['Type'] == 'Valley']

            print(f"\nSummary for {year}:")
            print(f"  Total Events: {len(events_df)}")
            print(f"  Peaks: {len(peaks)}")
            print(f"  Valleys: {len(valleys)}")
            print(f"  Average Daily Move: {events_df['DailyReturn'].abs().mean():.2f}%")
            print(f"  Largest Daily Move: {events_df['DailyReturn'].abs().max():.2f}%")

        print("-" * 80)

# Run the generation
generate_all_plotly_stories()

print("\n" + "=" * 80)
print("COMPLETE market story generation finished!")
print("All visible peaks and valleys have been detected and marked")
print("Interactive HTML files created - hover over markers for details")
print("=" * 80)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('cleaned_nifty50_data.csv')
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')

# ============================================================================
# FEATURE ENGINEERING - STOCK + NEWS HEADLINES
# ============================================================================

def prepare_data():
    """Prepare features combining stock data and news headlines"""
    print("🔧 Preparing data with Stock Prices + News Headlines features...")

    # Create daily aggregated data
    daily_data = df.groupby('Date').agg({
        'Daily_Return': 'mean',
        'HeadlinesCount': 'sum',
        'Volume_M': 'sum',
        'Price_Range_Pct': 'mean',
        'Close': 'mean',
        'Open': 'mean',
        'High': 'mean',
        'Low': 'mean'
    }).reset_index().sort_values('Date')

    # === STOCK TECHNICAL INDICATORS ===
    daily_data['Return_MA_5'] = daily_data['Daily_Return'].rolling(5).mean()
    daily_data['Return_MA_10'] = daily_data['Daily_Return'].rolling(10).mean()
    daily_data['Return_MA_20'] = daily_data['Daily_Return'].rolling(20).mean()
    daily_data['Volatility_5'] = daily_data['Daily_Return'].rolling(5).std()
    daily_data['Volatility_10'] = daily_data['Daily_Return'].rolling(10).std()
    daily_data['Volatility_20'] = daily_data['Daily_Return'].rolling(20).std()

    # Volume indicators
    daily_data['Volume_MA_5'] = daily_data['Volume_M'].rolling(5).mean()
    daily_data['Volume_MA_20'] = daily_data['Volume_M'].rolling(20).mean()
    daily_data['Volume_Ratio'] = daily_data['Volume_M'] / (daily_data['Volume_MA_20'] + 1)

    # Price momentum
    daily_data['Momentum_5'] = daily_data['Close'].pct_change(5) * 100
    daily_data['Momentum_10'] = daily_data['Close'].pct_change(10) * 100

    # RSI
    daily_data['RSI'] = calculate_rsi(daily_data['Daily_Return'])

    # === NEWS HEADLINES FEATURES ===
    # News volume features
    daily_data['News_MA_5'] = daily_data['HeadlinesCount'].rolling(5).mean()
    daily_data['News_MA_10'] = daily_data['HeadlinesCount'].rolling(10).mean()
    daily_data['News_MA_20'] = daily_data['HeadlinesCount'].rolling(20).mean()

    # News momentum (change in news volume)
    daily_data['News_Momentum'] = daily_data['HeadlinesCount'].diff()
    daily_data['News_Momentum_5'] = daily_data['HeadlinesCount'].diff(5)

    # News volatility (how much news volume fluctuates)
    daily_data['News_Volatility_5'] = daily_data['HeadlinesCount'].rolling(5).std()
    daily_data['News_Volatility_10'] = daily_data['HeadlinesCount'].rolling(10).std()

    # News intensity ratio
    daily_data['News_Intensity'] = daily_data['HeadlinesCount'] / (daily_data['News_MA_20'] + 1)

    # === INTERACTION FEATURES (NEWS + STOCK) ===
    # News impact on returns
    daily_data['News_Return_Interaction'] = daily_data['HeadlinesCount'] * daily_data['Daily_Return']
    daily_data['News_Volume_Interaction'] = daily_data['HeadlinesCount'] * daily_data['Volume_M']
    daily_data['News_Volatility_Interaction'] = daily_data['HeadlinesCount'] * daily_data['Volatility_5']

    # High news activity indicator
    daily_data['High_News_Activity'] = (daily_data['HeadlinesCount'] > daily_data['News_MA_20']).astype(int)

    # === LAGGED FEATURES (TEMPORAL) ===
    for lag in [1, 2, 3, 5]:
        # Stock lags
        daily_data[f'Return_Lag_{lag}'] = daily_data['Daily_Return'].shift(lag)
        daily_data[f'Volume_Lag_{lag}'] = daily_data['Volume_M'].shift(lag)
        daily_data[f'Volatility_Lag_{lag}'] = daily_data['Volatility_5'].shift(lag)

        # News lags
        daily_data[f'News_Lag_{lag}'] = daily_data['HeadlinesCount'].shift(lag)
        daily_data[f'NewsIntensity_Lag_{lag}'] = daily_data['News_Intensity'].shift(lag)

    # === TARGET VARIABLE ===
    # Predict next day return
    daily_data['Target'] = daily_data['Daily_Return'].shift(-1)

    # Drop rows with NaN
    daily_data = daily_data.dropna()

    print(f"✅ Data prepared: {len(daily_data)} samples with {len([c for c in daily_data.columns if c not in ['Date', 'Target']])} features")
    print(f"📰 News-related features: {len([c for c in daily_data.columns if 'News' in c or 'news' in c])}")

    return daily_data

def calculate_rsi(returns, periods=14):
    """Calculate Relative Strength Index"""
    delta = returns.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=periods).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=periods).mean()
    rs = gain / (loss + 1e-10)
    rsi = 100 - (100 / (1 + rs))
    return rsi

# ============================================================================
# MULTIPLE ML MODELS TRAINING
# ============================================================================

def train_multiple_models(X_train, X_test, y_train, y_test):
    """Train multiple ML models and compare performance"""
    print("\n🤖 Training Multiple ML Models...")
    print("=" * 70)

    # Define models - Only the 4 selected models
    models = {
        'Linear Regression': LinearRegression(),
        'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, max_depth=5, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, max_depth=5, random_state=42)
    }

    results = []
    trained_models = {}

    for name, model in models.items():
        print(f"\n📊 Training {name}...")

        # Train model
        model.fit(X_train, y_train)

        # Predictions
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        # Calculate metrics
        train_r2 = r2_score(y_train, y_pred_train)
        test_r2 = r2_score(y_test, y_pred_test)
        mae = mean_absolute_error(y_test, y_pred_test)
        mse = mean_squared_error(y_test, y_pred_test)
        rmse = np.sqrt(mse)

        # Accuracy within threshold
        accuracy_05 = np.mean(np.abs(y_test - y_pred_test) <= 0.5) * 100
        accuracy_10 = np.mean(np.abs(y_test - y_pred_test) <= 1.0) * 100

        results.append({
            'Model': name,
            'Train R²': train_r2,
            'Test R²': test_r2,
            'MAE': mae,
            'MSE': mse,
            'RMSE': rmse,
            'Accuracy (±0.5%)': accuracy_05,
            'Accuracy (±1.0%)': accuracy_10
        })

        trained_models[name] = {
            'model': model,
            'y_pred_train': y_pred_train,
            'y_pred_test': y_pred_test
        }

        print(f"   ✓ Test R² Score: {test_r2:.4f}")
        print(f"   ✓ MAE: {mae:.4f}")
        print(f"   ✓ RMSE: {rmse:.4f}")

    results_df = pd.DataFrame(results).sort_values('Test R²', ascending=False)

    print("\n" + "=" * 70)
    print("🏆 MODEL RANKINGS (by Test R²):")
    print("=" * 70)
    print(results_df.to_string(index=False))
    print("=" * 70)

    return results_df, trained_models

# ============================================================================
# VISUALIZATION - MODEL COMPARISON
# ============================================================================

def visualize_model_comparison(results_df, trained_models, X_test, y_test, feature_names):
    """Create comprehensive visualization of all models"""
    print("\n📊 Creating visualization...")

    # Get top 5 models
    top_models = results_df.head(5)['Model'].values

    fig = plt.figure(figsize=(24, 20))
    gs = fig.add_gridspec(4, 3, hspace=0.3, wspace=0.3)

    fig.suptitle('🎯 ML Models Performance: Stock Prices + News Headlines Impact Prediction',
                 fontsize=22, fontweight='bold', y=0.995)

    # ========== ROW 1: METRICS COMPARISON ==========

    # Plot 1: R² Score Comparison
    ax1 = fig.add_subplot(gs[0, 0])
    models_sorted = results_df.sort_values('Test R²', ascending=True)
    colors = ['#2ecc71' if r2 > 0.5 else '#e74c3c' if r2 < 0 else '#f39c12'
              for r2 in models_sorted['Test R²']]
    bars = ax1.barh(models_sorted['Model'], models_sorted['Test R²'], color=colors, alpha=0.8)
    ax1.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('R² Score', fontweight='bold', fontsize=11)
    ax1.set_title('Test R² Score Comparison', fontweight='bold', fontsize=13)
    ax1.grid(axis='x', alpha=0.3)
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax1.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                f'{width:.3f}', va='center', fontsize=9, fontweight='bold')

    # Plot 2: MAE Comparison
    ax2 = fig.add_subplot(gs[0, 1])
    models_sorted_mae = results_df.sort_values('MAE', ascending=True)
    bars = ax2.barh(models_sorted_mae['Model'], models_sorted_mae['MAE'],
                    color='#3498db', alpha=0.8)
    ax2.set_xlabel('Mean Absolute Error', fontweight='bold', fontsize=11)
    ax2.set_title('MAE Comparison (Lower is Better)', fontweight='bold', fontsize=13)
    ax2.grid(axis='x', alpha=0.3)
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax2.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                f'{width:.3f}', va='center', fontsize=9, fontweight='bold')

    # Plot 3: RMSE Comparison
    ax3 = fig.add_subplot(gs[0, 2])
    models_sorted_rmse = results_df.sort_values('RMSE', ascending=True)
    bars = ax3.barh(models_sorted_rmse['Model'], models_sorted_rmse['RMSE'],
                    color='#e67e22', alpha=0.8)
    ax3.set_xlabel('Root Mean Squared Error', fontweight='bold', fontsize=11)
    ax3.set_title('RMSE Comparison (Lower is Better)', fontweight='bold', fontsize=13)
    ax3.grid(axis='x', alpha=0.3)
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax3.text(width + 0.01, bar.get_y() + bar.get_height()/2,
                f'{width:.3f}', va='center', fontsize=9, fontweight='bold')

    # ========== ROW 2: ACTUAL VS PREDICTED (TOP 3 MODELS) ==========

    for idx, model_name in enumerate(top_models[:3]):
        ax = fig.add_subplot(gs[1, idx])
        y_pred = trained_models[model_name]['y_pred_test']

        ax.scatter(y_test, y_pred, alpha=0.5, s=30, color='#3498db', edgecolors='white', linewidth=0.5)

        # Perfect prediction line
        min_val = min(y_test.min(), y_pred.min())
        max_val = max(y_test.max(), y_pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2,
                label='Perfect Prediction', alpha=0.8)

        # Trend line
        z = np.polyfit(y_test, y_pred, 1)
        p = np.poly1d(z)
        ax.plot(y_test.sort_values(), p(y_test.sort_values()), "g-",
                linewidth=2, label='Trend Line', alpha=0.7)

        r2 = results_df[results_df['Model'] == model_name]['Test R²'].values[0]
        mae = results_df[results_df['Model'] == model_name]['MAE'].values[0]

        ax.set_xlabel('Actual Returns (%)', fontweight='bold', fontsize=10)
        ax.set_ylabel('Predicted Returns (%)', fontweight='bold', fontsize=10)
        ax.set_title(f'{model_name}\nR²={r2:.3f}, MAE={mae:.3f}',
                    fontweight='bold', fontsize=12)
        ax.legend(loc='upper left', fontsize=8)
        ax.grid(True, alpha=0.3)

    # ========== ROW 3: PREDICTION ERRORS & ACCURACY ==========

    # Plot 7: Error Distribution (Best Model)
    ax7 = fig.add_subplot(gs[2, 0])
    best_model_name = results_df.iloc[0]['Model']
    y_pred_best = trained_models[best_model_name]['y_pred_test']
    errors = y_test - y_pred_best

    ax7.hist(errors, bins=50, color='#9b59b6', alpha=0.7, edgecolor='black')
    ax7.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    ax7.set_xlabel('Prediction Error (%)', fontweight='bold', fontsize=10)
    ax7.set_ylabel('Frequency', fontweight='bold', fontsize=10)
    ax7.set_title(f'Error Distribution - {best_model_name}', fontweight='bold', fontsize=12)
    ax7.legend()
    ax7.grid(axis='y', alpha=0.3)

    # Plot 8: Accuracy Comparison
    ax8 = fig.add_subplot(gs[2, 1])
    accuracy_data = results_df[['Model', 'Accuracy (±0.5%)', 'Accuracy (±1.0%)']].head(8)
    x = np.arange(len(accuracy_data))
    width = 0.35

    bars1 = ax8.bar(x - width/2, accuracy_data['Accuracy (±0.5%)'], width,
                    label='±0.5%', color='#2ecc71', alpha=0.8)
    bars2 = ax8.bar(x + width/2, accuracy_data['Accuracy (±1.0%)'], width,
                    label='±1.0%', color='#3498db', alpha=0.8)

    ax8.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=10)
    ax8.set_title('Prediction Accuracy (Top 8 Models)', fontweight='bold', fontsize=12)
    ax8.set_xticks(x)
    ax8.set_xticklabels(accuracy_data['Model'], rotation=45, ha='right', fontsize=8)
    ax8.legend()
    ax8.grid(axis='y', alpha=0.3)

    # Plot 9: Train vs Test R² (Overfitting Check)
    ax9 = fig.add_subplot(gs[2, 2])
    x_pos = np.arange(len(results_df))
    width = 0.35

    bars1 = ax9.bar(x_pos - width/2, results_df['Train R²'], width,
                    label='Train R²', color='#1abc9c', alpha=0.8)
    bars2 = ax9.bar(x_pos + width/2, results_df['Test R²'], width,
                    label='Test R²', color='#e74c3c', alpha=0.8)

    ax9.set_ylabel('R² Score', fontweight='bold', fontsize=10)
    ax9.set_title('Train vs Test R² (Overfitting Check)', fontweight='bold', fontsize=12)
    ax9.set_xticks(x_pos)
    ax9.set_xticklabels(results_df['Model'], rotation=45, ha='right', fontsize=8)
    ax9.legend()
    ax9.grid(axis='y', alpha=0.3)
    ax9.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

    # ========== ROW 4: FEATURE IMPORTANCE & TIME SERIES ==========

    # Plot 10: Feature Importance (Best Model)
    ax10 = fig.add_subplot(gs[3, 0])
    best_model = trained_models[best_model_name]['model']

    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        indices = np.argsort(importances)[-15:]  # Top 15

        colors_feat = ['#e74c3c' if 'News' in feature_names[i] or 'news' in feature_names[i]
                       else '#3498db' for i in indices]

        ax10.barh(range(len(indices)), importances[indices], color=colors_feat, alpha=0.8)
        ax10.set_yticks(range(len(indices)))
        ax10.set_yticklabels([feature_names[i] for i in indices], fontsize=8)
        ax10.set_xlabel('Importance Score', fontweight='bold', fontsize=10)
        ax10.set_title(f'Top 15 Feature Importance - {best_model_name}\n(Red=News, Blue=Stock)',
                      fontweight='bold', fontsize=12)
        ax10.grid(axis='x', alpha=0.3)
    else:
        ax10.text(0.5, 0.5, 'Feature importance\nnot available\nfor this model',
                 ha='center', va='center', fontsize=12, transform=ax10.transAxes)
        ax10.set_title(f'Feature Importance - {best_model_name}', fontweight='bold', fontsize=12)

    # Plot 11: Residuals Plot
    ax11 = fig.add_subplot(gs[3, 1])
    residuals = errors
    ax11.scatter(y_pred_best, residuals, alpha=0.5, s=30, color='#9b59b6',
                edgecolors='white', linewidth=0.5)
    ax11.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax11.set_xlabel('Predicted Returns (%)', fontweight='bold', fontsize=10)
    ax11.set_ylabel('Residuals (%)', fontweight='bold', fontsize=10)
    ax11.set_title(f'Residuals Plot - {best_model_name}', fontweight='bold', fontsize=12)
    ax11.grid(True, alpha=0.3)

    # Plot 12: Model Performance Radar
    ax12 = fig.add_subplot(gs[3, 2], projection='polar')

    # Normalize metrics for radar chart (top 5 models)
    metrics_to_plot = ['Test R²', 'Accuracy (±0.5%)', 'Accuracy (±1.0%)']
    angles = np.linspace(0, 2 * np.pi, len(metrics_to_plot), endpoint=False).tolist()
    angles += angles[:1]

    for i, model_name in enumerate(top_models[:5]):
        model_data = results_df[results_df['Model'] == model_name].iloc[0]
        values = [
            max(0, model_data['Test R²']),
            model_data['Accuracy (±0.5%)'] / 100,
            model_data['Accuracy (±1.0%)'] / 100
        ]
        values += values[:1]

        ax12.plot(angles, values, 'o-', linewidth=2, label=model_name, alpha=0.7)
        ax12.fill(angles, values, alpha=0.15)

    ax12.set_xticks(angles[:-1])
    ax12.set_xticklabels(metrics_to_plot, fontsize=9)
    ax12.set_ylim(0, 1)
    ax12.set_title('Top 5 Models - Performance Radar', fontweight='bold',
                  fontsize=12, pad=20)
    ax12.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=8)
    ax12.grid(True)

    plt.savefig('ml_models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    print("✅ Visualization saved as 'ml_models_comparison.png'")

# ============================================================================
# NEWS IMPACT ANALYSIS
# ============================================================================

def analyze_news_impact(trained_models, results_df, daily_data, feature_names):
    """Analyze how news headlines affect predictions"""
    print("\n📰 Analyzing News Headlines Impact on Stock Predictions...")

    # Get best model
    best_model_name = results_df.iloc[0]['Model']
    best_model = trained_models[best_model_name]['model']

    # Identify news-related features
    news_features = [f for f in feature_names if 'News' in f or 'news' in f]

    print(f"\n📊 News-related features used in modeling: {len(news_features)}")
    for feat in news_features:
        print(f"   • {feat}")

    # Feature importance for news features (if available)
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        news_importance = {}

        for feat in news_features:
            if feat in feature_names:
                idx = feature_names.index(feat)
                news_importance[feat] = importances[idx]

        if news_importance:
            print(f"\n🎯 News Feature Importance in {best_model_name}:")
            for feat, imp in sorted(news_importance.items(), key=lambda x: x[1], reverse=True):
                print(f"   {feat}: {imp:.4f}")

            total_importance = sum(importances)
            news_total_importance = sum(news_importance.values())
            news_percentage = (news_total_importance / total_importance) * 100

            print(f"\n📈 Total contribution of NEWS features: {news_percentage:.2f}%")
            print(f"📈 Total contribution of STOCK features: {100-news_percentage:.2f}%")

# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main():
    """Main execution function"""
    print("="*70)
    print("🚀 STOCK MARKET PREDICTION WITH NEWS HEADLINES IMPACT")
    print("="*70)

    # Prepare data
    daily_data = prepare_data()

    # Define features and target
    exclude_cols = ['Date', 'Target', 'Close', 'Open', 'High', 'Low']
    feature_cols = [col for col in daily_data.columns if col not in exclude_cols]

    X = daily_data[feature_cols]
    y = daily_data['Target']

    print(f"\n📊 Dataset Summary:")
    print(f"   Total Samples: {len(X)}")
    print(f"   Total Features: {len(feature_cols)}")
    print(f"   Target Variable: Next Day Return (%)")

    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, shuffle=False
    )

    print(f"\n📂 Data Split:")
    print(f"   Training Set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
    print(f"   Test Set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train models
    results_df, trained_models = train_multiple_models(
        X_train_scaled, X_test_scaled, y_train, y_test
    )

    # Visualize results
    visualize_model_comparison(results_df, trained_models, X_test_scaled, y_test, feature_cols)

    # Analyze news impact
    analyze_news_impact(trained_models, results_df, daily_data, feature_cols)

    print("\n" + "="*70)
    print("✅ ANALYSIS COMPLETE!")
    print("="*70)
    print("\n📊 Key Findings:")
    print(f"   🥇 Best Model: {results_df.iloc[0]['Model']}")
    print(f"   📈 Best R² Score: {results_df.iloc[0]['Test R²']:.4f}")
    print(f"   📉 Best MAE: {results_df.iloc[0]['MAE']:.4f}")
    print(f"   🎯 Best RMSE: {results_df.iloc[0]['RMSE']:.4f}")
    print("\n💡 The models predict how NEWS HEADLINES affect stock price movements!")
    print("="*70)

if __name__ == "__main__":
    main()

## Optional Streamlit deployment

The original notebook referenced an external `aap.py` Streamlit application that was not included in the uploaded files. Therefore, this repository does not claim to contain a runnable dashboard. If you add the application source later, install Streamlit and run it locally with `streamlit run app.py`.


**Security note:** Do not commit ngrok authentication tokens, API keys, or other credentials. Configure secrets locally or through your deployment provider's secret manager.
